# Protein abundance localizes continuous-time quantum walks on mitochondrial pathway networks
### Analysis pipeline — version 3, revised in response to peer review

**Discover Applied Sciences, submission 656b5d97-9255-41a0-9d05-ee68ee016154.**
This version implements every reviewer request that required new computation. Changes with
respect to the submitted version:

| § | Addition | Reviewer point |
|---|---|---|
| 1.6, 9.7 | Imputation re-grounded on the observed detection limit; five schemes compared; Hermiticity statement corrected | R3.4 |
| 5.4 | Primary endpoint at 5 000 permutations; exact empirical *p*-value formula documented | R1.4, R3.5 |
| 5.6, 5.7 | Decomposition of localisation into topology, off-diagonal and diagonal contributions; falsifiable test of the Anderson signature | R1.3, R3.2 |
| 5.8 | Observation window, time grid and integration method fully specified and varied | R3.6 |
| 6.2 | Complete Lindblad parameter set; convergence in time step and total time | R3.10 |
| 7.3, 7.4 | Disease-gene enrichment adjusted for pathway membership, pathway degree and abundance; stratified permutation; multiplicity correction | R1.5, R3.8, R3.9 |
| 9.6 | Four abundance representations compared | R3.3 |
| 9.8–9.10 | Benchmark against eight non-quantum comparators on two tasks, same data and criteria | R2 |

Textual requests (biological interpretation of propagation, Hamiltonian rationale, neutral
terminology, removal of editorial judgements, repository link) are addressed in the manuscript.

**Runtime** ≈ 35–50 min, dominated by the permutation tests. Set `N_PERMUTATIONS` to 300 and
`N_PERM_CONTROL` to 200 for a fast structural check before a full run.

**Data** — place `Human_MitoCarta3_0.xls` beside this notebook. Requires `xlrd`, `statsmodels`,
`networkx` and `scikit-learn`.


---
## ⚠️ Cadrage v2 — ce que les données réelles autorisent à écrire

Cette version est révisée **après une première exécution complète sur MitoCarta 3.0**. Les résultats
obtenus invalident une partie du plan initial. Les conserver tels quels exposerait à un rejet ; ce
notebook les traite donc explicitement.

**Trois résultats négatifs à assumer dans le manuscrit** (les cacher est le meilleur moyen d'être rejeté) :

1. **L'avantage quantique est < 1 dans les 14 tissus.** La marche classique atteint la voie cible plus
   efficacement. Aucune formulation du type *quantum-enhanced transport* n'est soutenable.
2. **Pas de transport assisté par l'environnement.** La courbe $\eta(\gamma)$ décroît de façon monotone :
   aucun optimum de déphasage, donc aucune signature ENAQT. Section reclassée en **contrôle négatif**.
3. **Le critère par paire (Complexe I → Complexe V) n'a aucun support statistique** : $q = 0.77$ pour les
   14 tissus, nul topologique $p = 0.23$, classement non reproductible entre variantes de modélisation
   ($\rho \approx 0.2$). Reclassé en illustration mécanistique.

**Le résultat qui tient — et qui devient le cœur de l'article :**

> L'hétérogénéité d'abondance protéique agit comme un **désordre diagonal** qui **localise** la marche
> quantique sur le réseau des voies mitochondriales. L'IPR quantique plafonne autour de 20–30 voies
> quand la diffusion classique en explore plus de 120. Le degré de localisation est **tissu-spécifique**
> et significativement supérieur au hasard, à distribution d'intensités constante, dans les tissus à
> forte spécialisation oxydative.

Ce cadrage explique d'un seul coup les trois résultats négatifs : en régime localisé, la marche
cohérente *doit* transporter moins bien que la diffusion classique, et le déphasage ne peut que dégrader
un transport déjà limité par le piégeage. C'est cohérent, testable, et c'est de la physique connue
(localisation d'Anderson) appliquée pour la première fois à un réseau protéomique tissulaire réel.

**Deux règles méthodologiques qui en découlent**, appliquées partout dans ce notebook :

- **Aucune comparaison inter-tissus sur une valeur brute.** La couverture MS varie de 15 % à 44 % de
  protéines non détectées selon le tissu, et explique ~87 % de la variance du classement brut de
  conductivité ($\rho = -0.93$). Le critère rapporté est un **z-score contre un nul apparié au tissu**
  (§5.4), qui conserve la distribution marginale des intensités et n'est plus confondu ($\rho = -0.20$, n.s.).
- **Tout résultat principal est répliqué sans imputation** sur les protéines détectées dans les 14 tissus
  (§9.5), et avec la diagonale désactivée pour isoler la contribution du désordre de site.


## 0. Configuration globale

In [ ]:
# ============================================================================
# Paramètres du pipeline — tout est modifiable ici, rien n'est codé en dur ailleurs
# ============================================================================
CONFIG = {
    # --- données -------------------------------------------------------------
    "DATA_PATH":            None,       # renseigné par la cellule d'import §1.0
    "SHEET_A_HINT":         "A Human",  # inventaire (PAS "B Human All Genes")
    "SHEET_C_HINT":         "MitoPathways",
    "BACKGROUND":           0.1,        # valeur heritee de v1 ; remplacee au §1.6 (R3.4)
    "LOG_ALREADY":          True,       # les colonnes MitoCarta sont déjà en log10

    # --- topologie du réseau -------------------------------------------------
    "DROP_UNQUANTIFIED":     True,      # écarter les protéines sans aucune mesure MS (1136 -> ~957)
    "MIN_GENES_PER_PATHWAY": 3,         # voies trop petites = bruit ; mettre 1 pour garder les 149 voies
    "MIN_SHARED_GENES":      1,         # nb minimal de gènes partagés pour créer une arête
    "KEEP_LEAVES_ONLY":      False,     # True = ne garder que les voies terminales de la hiérarchie

    # --- hamiltonien ---------------------------------------------------------
    "INCLUDE_DIAGONAL":      True,      # énergies de site H_ii = abondance moyenne de la voie
    "NORMALIZE":             "spectral",# 'spectral' (||H||_2 = 1) | 'none'

    # --- dynamique -----------------------------------------------------------
    "T_MAX":                 60.0,
    "N_T":                   601,
    "T_STAR":                10.0,      # instant de référence pour U(t*) (clustering)
    "SINK_RATE":             1.0,       # Γ du puits absorbant (modèle de piégeage)
    "SOURCE_PATHWAY":        "Complex I",
    "TARGET_PATHWAY":        "Complex V",

    # --- statistiques --------------------------------------------------------
    "N_PERMUTATIONS":        5000,  # critere principal (R1.4, R3.5)
    "N_PERM_CONTROL":        2000,  # configurations de controle
    "N_PERM_BENCHMARK":      2000,  # comparaison aux methodes non quantiques
    "N_PERM_STRATIFIED":     5000,  # permutation stratifiee (enrichissement)
    "N_BOOTSTRAP_AUC":       2000,  # bootstrap apparie sur les ecarts d'AUROC
    "N_BOOTSTRAP":           200,
    "SEED":                  20260805,

    # --- sorties -------------------------------------------------------------
    "OUTDIR":                "results",
    "FIG_DPI":               300,
    "FIG_FORMATS":           ("png", "pdf"),
}

In [ ]:
import os, re, sys, json, time, math, platform, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import scipy.linalg as sla
from scipy.sparse.csgraph import connected_components
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --- compatibilité des versions (numpy 1.x/2.x, pandas 2.x/3.x) --------------
TRAPZ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz   # np.trapz -> np.trapezoid (numpy 2)
def elementwise(df, fn):
    """df.map (pandas >= 2.1) ou df.applymap (pandas < 2.1)."""
    return df.map(fn) if hasattr(df, "map") and not isinstance(df, pd.Series) else df.applymap(fn)
def spearman(a, b):
    r = spearmanr(a, b)
    return float(getattr(r, "statistic", r[0]))

RNG = np.random.default_rng(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

OUT = Path(CONFIG["OUTDIR"])
FIGDIR, TABDIR = OUT / "figures", OUT / "tables"
for d in (FIGDIR, TABDIR):
    d.mkdir(parents=True, exist_ok=True)

# Style de figure publication-ready (Springer Nature : 300 dpi min, police sans-serif)
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": CONFIG["FIG_DPI"], "savefig.bbox": "tight",
    "font.family": "sans-serif", "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "lines.linewidth": 1.4,
})

def savefig(fig, name, caption=""):
    """Sauvegarde une figure dans tous les formats demandés et journalise la légende."""
    for ext in CONFIG["FIG_FORMATS"]:
        fig.savefig(FIGDIR / f"{name}.{ext}")
    FIGURE_INDEX.append({"figure": name, "caption": caption,
                         "files": [f"{name}.{e}" for e in CONFIG["FIG_FORMATS"]]})
    return fig

FIGURE_INDEX = []
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)


---
## 1. Import de la base de données MitoCarta 3.0 et contrôle qualité

**Rien n'est simulé ici.** Le pipeline lit l'inventaire expérimental MitoCarta 3.0 :

| Feuille | Contenu utilisé |
|---|---|
| `A Human MitoCarta3.0` | 1 136 protéines, colonne `Symbol`, 14 colonnes `*_total_peak_intensity_log10` |
| `C MitoPathways` | 149 voies, colonnes `MitoPathways Hierarchy` et `Genes` |

⚠️ La feuille `B Human All Genes` (19 247 gènes, dont non-mitochondriaux) est **explicitement écartée** :
la confondre avec la feuille A gonflerait le réseau de milliers de protéines non mitochondriales.

In [ ]:
# ============================================================================
# 1.0  ▶ IMPORT DE LA BASE DE DONNÉES — EXÉCUTER CETTE CELLULE EN PREMIER
# ============================================================================
# Fichier attendu : Human_MitoCarta3_0.xls (ou Human.MitoCarta3.0.xls), Broad Institute.
# https://www.broadinstitute.org/mitocarta/mitocarta30-inventory-mammalian-mitochondrial-proteins-and-pathways
#
# Ordre des tentatives : (1) chemin fourni  (2) fichier déjà présent sur le disque
# (3) sélecteur interactif (Google Colab / boîte de dialogue) (4) saisie manuelle du chemin.
import glob, hashlib, shutil

PATTERNS = ["Human*MitoCarta3*.xls*", "*MitoCarta*.xls*", "data/*MitoCarta*.xls*",
            "../*MitoCarta*.xls*", "/content/*MitoCarta*.xls*",
            str(Path.home() / "Downloads" / "*MitoCarta*.xls*"),
            str(Path.home() / "Desktop" / "*MitoCarta*.xls*")]


def _check_xlrd():
    try:
        import xlrd                                                   # noqa: F401
    except ImportError:
        raise ImportError("Le format .xls exige xlrd :  pip install xlrd\n"
                          "(alternative : ré-enregistrer le classeur en .xlsx depuis Excel)")


def _find_local():
    hits = []
    for pat in PATTERNS:
        hits += glob.glob(pat)
    hits = [h for h in dict.fromkeys(hits) if Path(h).is_file()]
    return sorted(hits, key=lambda f: -Path(f).stat().st_size)


def _upload_colab():
    from google.colab import files                                    # noqa
    print("→ Sélectionnez Human_MitoCarta3_0.xls dans la fenêtre d'import…")
    up = files.upload()
    return list(up)[0] if up else None


def _upload_dialog():
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
    f = filedialog.askopenfilename(title="Sélectionner Human_MitoCarta3_0.xls",
                                   filetypes=[("Classeurs Excel", "*.xls *.xlsx"), ("Tous", "*.*")])
    root.destroy()
    return f or None


def import_database(path=None, copy_to="data"):
    '''Localise/importe le classeur MitoCarta 3.0 et renvoie son chemin validé.'''
    _check_xlrd()
    candidate = None

    if path and Path(path).is_file():
        candidate = str(path)
    if candidate is None:
        local = _find_local()
        if local:
            candidate = local[0]
            print(f"Fichier détecté automatiquement : {candidate}")
    if candidate is None and "google.colab" in sys.modules:
        candidate = _upload_colab()
    if candidate is None:
        try:
            candidate = _upload_dialog()
        except Exception:
            pass
    if candidate is None:
        try:
            candidate = input("Chemin complet vers Human_MitoCarta3_0.xls : ").strip().strip('\"')
        except Exception:
            candidate = None
    if not candidate or not Path(candidate).is_file():
        raise FileNotFoundError(
            "Base de données introuvable.\n"
            "Placez Human_MitoCarta3_0.xls à côté de ce notebook, ou appelez :\n"
            "    import_database('/chemin/vers/Human_MitoCarta3_0.xls')")

    src = Path(candidate)
    if copy_to:
        Path(copy_to).mkdir(exist_ok=True)
        dst = Path(copy_to) / src.name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst); src = dst

    digest = hashlib.sha256(src.read_bytes()).hexdigest()
    CONFIG["DATA_PATH"] = str(src)
    globals()["PROVENANCE"] = {"file": src.name, "abspath": str(src.resolve()),
                               "size_MB": round(src.stat().st_size / 1e6, 2),
                               "sha256": digest, "source": "MitoCarta3.0, Broad Institute"}
    print(f"✔ Base importée : {src}  ({PROVENANCE['size_MB']} Mo)")
    print(f"  SHA-256 : {digest[:32]}…   (à citer dans Data Availability)")
    return str(src)


DATA_FILE = import_database()      # ← ou : import_database('/chemin/Human_MitoCarta3_0.xls')

In [ ]:
# ============================================================================
# 1.1 Lecture du classeur — sélection stricte des feuilles A et C
# ============================================================================
TISSUE_RE = re.compile(r"^(.*?)_total_peak_intensity_log10$", re.I)


def _read_workbook(path):
    path = Path(path)
    engine = "xlrd" if path.suffix.lower() == ".xls" else "openpyxl"
    try:
        return pd.read_excel(path, sheet_name=None, engine=engine)
    except Exception:
        return pd.read_excel(path, sheet_name=None)


def _pick_sheets(sheets, cfg=CONFIG):
    '''Feuille A = inventaire mitochondrial (jamais "B ... All Genes") ; feuille C = MitoPathways.'''
    names = list(sheets)
    sheet_a = next((n for n in names if cfg["SHEET_A_HINT"].lower() in n.lower()), None)
    if sheet_a is None:                       # repli : feuille avec colonnes tissus, hors 'all genes'
        cands = [n for n in names
                 if any(TISSUE_RE.match(str(c)) for c in sheets[n].columns)
                 and "all gene" not in n.lower()]
        sheet_a = min(cands, key=lambda n: len(sheets[n])) if cands else None
    sheet_c = next((n for n in names if cfg["SHEET_C_HINT"].lower() in n.lower()), None)
    if sheet_c is None:                       # repli : feuille aux listes de gènes séparées par virgules
        sheet_c = max((n for n in names if n != sheet_a),
                      key=lambda n: elementwise(sheets[n].astype(str), lambda v: str(v).count(",")).mean().max())
    if sheet_a is None or sheet_c is None:
        raise ValueError(f"Feuilles A/C non identifiées parmi {names}")
    return sheet_a, sheet_c


SHEETS = _read_workbook(CONFIG["DATA_PATH"])
SHEET_A, SHEET_C = _pick_sheets(SHEETS)
df_a, df_c = SHEETS[SHEET_A].copy(), SHEETS[SHEET_C].copy()

print(f"Feuilles du classeur : {list(SHEETS)}")
print(f"→ inventaire  : '{SHEET_A}'  {df_a.shape}")
print(f"→ MitoPathways: '{SHEET_C}'  {df_c.shape}")

# Garde-fous : on refuse de continuer sur une feuille manifestement incorrecte
assert "all gene" not in SHEET_A.lower(), "Feuille B sélectionnée par erreur (gènes non mitochondriaux)"
if not (900 <= len(df_a) <= 1500):
    print(f"[!] {len(df_a)} lignes dans la feuille A — attendu ≈1136 pour MitoCarta 3.0 humain. Vérifiez le fichier.")

In [ ]:
# ============================================================================
# 1.2 Détection des colonnes : symbole de gène, tissus, hiérarchie/gènes des voies
# ============================================================================
def detect_symbol_col(df):
    for cand in ("Symbol", "symbol", "Gene", "GeneSymbol", "HumanGeneSymbol"):
        if cand in df.columns:
            return cand
    return df.columns[0]


def detect_tissue_cols(df):
    cols = {}
    for c in df.columns:
        m = TISSUE_RE.match(str(c))
        if m:
            cols[m.group(1).strip().lower()] = c
    if not cols:  # repli : toute colonne contenant 'intensity'
        cols = {str(c).lower(): c for c in df.columns if "intensity" in str(c).lower()}
    return cols


def detect_pathway_cols(df):
    obj = df.astype(str)
    genes_col = elementwise(obj, lambda v: str(v).count(",")).mean().idxmax()
    rest = obj.drop(columns=[genes_col])
    arrows = elementwise(rest, lambda v: ">" in str(v)).mean()
    path_col = arrows.idxmax() if arrows.max() > 0 else rest.columns[0]
    return path_col, genes_col


SYMBOL_COL = detect_symbol_col(df_a)
TISSUE_COLS = detect_tissue_cols(df_a)
PATH_COL, GENES_COL = detect_pathway_cols(df_c)
TISSUES = list(TISSUE_COLS)

print(f"Colonne symbole      : {SYMBOL_COL}")
print(f"Colonnes hiérarchie/gènes (feuille C) : '{PATH_COL}' / '{GENES_COL}'")
print(f"{len(TISSUES)} tissus détectés : {', '.join(TISSUES)}")
assert len(TISSUES) >= 2, "Aucune colonne d'intensité tissulaire détectée — vérifiez le fichier."

In [ ]:
# ============================================================================
# 1.3 Matrice d'expression, imputation par le bruit de fond, QC
# ============================================================================
expr = df_a[[SYMBOL_COL] + [TISSUE_COLS[t] for t in TISSUES]].copy()
expr.columns = ["gene"] + TISSUES
expr["gene"] = expr["gene"].astype(str).str.strip()
expr = expr[expr["gene"].ne("") & expr["gene"].ne("nan")]
for t in TISSUES:
    expr[t] = pd.to_numeric(expr[t], errors="coerce")

expr = expr.groupby("gene", as_index=False)[TISSUES].mean()          # doublons de symboles
detected = expr[TISSUES].notna().any(axis=1)
qc = pd.DataFrame({
    "n_genes_sheetA": [len(expr)],
    "n_genes_quantified": [int(detected.sum())],
    "pct_missing_values": [100 * expr[TISSUES].isna().to_numpy().mean()],
    "n_tissues": [len(TISSUES)],
})
expr = expr[detected].reset_index(drop=True)

# Imputation : les valeurs manquantes = protéine non détectée -> bruit de fond mesurable.
# Choix conservateur qui préserve la positivité des poids et donc l'hermiticité de H.
expr_imputed = expr.copy()
expr_imputed[TISSUES] = expr_imputed[TISSUES].fillna(CONFIG["BACKGROUND"])
expr_imputed[TISSUES] = expr_imputed[TISSUES].clip(lower=CONFIG["BACKGROUND"])

GENES = expr_imputed["gene"].to_numpy()
X = expr_imputed[TISSUES].to_numpy(float)          # (n_genes × n_tissues)
GENE_INDEX = {g: i for i, g in enumerate(GENES)}

qc.to_csv(TABDIR / "qc_expression.csv", index=False)
display(qc)
display(expr_imputed[TISSUES].describe().T[["mean", "std", "min", "max"]].round(2))

In [ ]:
# --- Figure QC : distribution des intensités par tissu -----------------------
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.boxplot([X[:, j] for j in range(len(TISSUES))], showfliers=False, widths=0.6)
ax.set_xticks(range(1, len(TISSUES) + 1)); ax.set_xticklabels(TISSUES)
ax.set_ylabel(r"MS intensity ($\log_{10}$)")
ax.set_title("Mitochondrial protein abundance by tissue")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
savefig(fig, "figS1_qc_intensities", "Distribution of mass-spectrometric intensities by tissue.")
plt.show()

In [ ]:
# ============================================================================
# 1.4 Profondeur de détection par tissu — variable de confusion n°1
# ============================================================================
# La couverture MS varie fortement d'un tissu à l'autre. Toute métrique calculée
# sur les intensités imputées hérite mécaniquement de cette variation : elle doit
# donc être mesurée ici, puis testée contre chaque résultat (§5.5).
X_RAW = expr[TISSUES].to_numpy(float)          # avant imputation (NaN conservés)
MISSING_FRAC = np.isnan(X_RAW).mean(0)

depth = pd.DataFrame({
    "tissue": TISSUES,
    "n_detected": (~np.isnan(X_RAW)).sum(0),
    "missing_frac": MISSING_FRAC,
    "mean_intensity_detected": np.nanmean(X_RAW, axis=0),
}).sort_values("missing_frac", ascending=False).reset_index(drop=True)
depth.to_csv(TABDIR / "detection_depth.csv", index=False)
display(depth.round(3))

print(f"Écart de couverture : {100*MISSING_FRAC.min():.1f} % à {100*MISSING_FRAC.max():.1f} % de protéines non détectées.")
print("→ Aucune comparaison inter-tissus de valeur BRUTE n'est interprétable sans contrôle de ce facteur.\n"
      "  Le critère retenu (§5.4) est un z-score contre un nul apparié au tissu, donc insensible\n"
      "  à la distribution marginale des intensités.")


In [ ]:
# ============================================================================
# 1.5 Jeu « complete-case » — protéines détectées dans les 14 tissus, SANS imputation
# ============================================================================
# Piste de contrôle : si le résultat principal survit ici, l'objection « artefact
# d'imputation » tombe. Le prix à payer est une perte d'effectif (~60 %).
COMPLETE = ~np.isnan(X_RAW).any(axis=1)
X_CC_FULL = X_RAW[COMPLETE]                      # aucune valeur imputée
GENES_CC = GENES[COMPLETE]

print(f"Protéines détectées dans les 14 tissus : {COMPLETE.sum()} / {len(GENES)} "
      f"({100*COMPLETE.mean():.1f} %)")
print(f"Intensité : min {X_CC_FULL.min():.2f} | médiane {np.median(X_CC_FULL):.2f} | max {X_CC_FULL.max():.2f}")
print("→ Le désordre diagonal résiduel provient ici de la seule biologie, plus du bruit de fond imposé.")


---
### 1.6 Schéma d'imputation — réponse au relecteur 3, point 4

Le relecteur demande d'où vient la valeur 0.1 et si elle correspond à un seuil de détection.
**Vérification faite sur le fichier source : non.** Les intensités réellement détectées dans
MitoCarta 3.0 s'étendent de **4.70 à 12.00** en $\log_{10}$, et aucune valeur inférieure ou
égale à 0.1 n'y figure. La valeur 0.1 était donc un choix numérique arbitraire, situé très
en dessous de toute mesure réelle, ce qui introduisait un désordre artificiel considérable.

La version révisée adopte comme référence une **imputation censurée à gauche à la limite de
détection propre à chaque tissu** (MinDet au sens de Lazar et al. 2016), et compare
systématiquement cinq schémas au §9.7.

**Correction d'une erreur mathématique.** Le manuscrit précédent affirmait que la positivité
des poids était nécessaire pour préserver l'hermiticité de $H$. C'est faux, et le relecteur a
raison de le relever : l'hermiticité découle de ce que $H$ est réelle et symétrique,
$H = (H + H^\top)/2$, indépendamment du signe des éléments. La positivité est une contrainte
d'interprétation, non d'algèbre. L'énoncé est corrigé partout.


In [ ]:
# ============================================================================
# 1.6 Schémas d'imputation — R3.4 : justification de la valeur de remplacement
# ============================================================================
# Fait factuel vérifié sur le fichier source : la plus petite intensité RÉELLEMENT
# détectée dans MitoCarta 3.0 vaut 4.70 (log10), et aucune valeur <= 0.1 n'existe.
# La valeur 0.1 utilisée en v1/v2 n'est donc PAS un bruit de fond mesuré : c'est un
# choix numérique. Le relecteur 3 a raison de le signaler. On construit ici cinq
# schémas, dont quatre reposent sur la limite de détection observée, et on les
# compare systématiquement (§9.7).
MISS = np.isnan(X_RAW)
MIN_DETECTED = float(np.nanmin(X_RAW))
TISSUE_MIN = np.nanmin(X_RAW, axis=0)
TISSUE_P1 = np.nanpercentile(X_RAW, 1, axis=0)
TISSUE_SD = np.nanstd(X_RAW, axis=0)

_rng_imp = np.random.default_rng(CONFIG["SEED"])
_minprob = X_RAW.copy()
for _j in range(X_RAW.shape[1]):
    _k = MISS[:, _j].sum()
    if _k:
        _minprob[MISS[:, _j], _j] = _rng_imp.normal(TISSUE_MIN[_j], 0.3 * TISSUE_SD[_j], _k)

IMPUTATIONS = {
    "legacy 0.1":      np.where(MISS, 0.1, X_RAW),
    "global minimum":  np.where(MISS, MIN_DETECTED, X_RAW),
    "per-tissue min":  np.where(MISS, np.tile(TISSUE_MIN, (X_RAW.shape[0], 1)), X_RAW),
    "MinDet (p1)":     np.where(MISS, np.tile(TISSUE_P1, (X_RAW.shape[0], 1)), X_RAW),
    "MinProb":         _minprob,
}

# --- Le schéma de référence de la version révisée ---------------------------
# « per-tissue minimum » : imputation censurée à gauche à la limite de détection
# propre à chaque tissu (MinDet au sens de Lazar et al. 2016). Défendable, et
# n'introduit aucune valeur hors de l'intervalle réellement observé.
PRIMARY_IMPUTATION = "per-tissue min"
X = IMPUTATIONS[PRIMARY_IMPUTATION].copy()

imp_summary = pd.DataFrame({
    "scheme": list(IMPUTATIONS),
    "imputed_value_range": [f"{v[MISS].min():.2f}–{v[MISS].max():.2f}" for v in IMPUTATIONS.values()],
    "within_observed_range": [bool(v[MISS].min() >= MIN_DETECTED - 1e-9) for v in IMPUTATIONS.values()],
})
imp_summary.to_csv(TABDIR / "imputation_schemes.csv", index=False)
display(imp_summary)

print(f"Intensités détectées : min {MIN_DETECTED:.2f}, max {np.nanmax(X_RAW):.2f} (log10)")
print(f"Aucune valeur <= 0.1 n'existe dans les données ({int((X_RAW <= 0.1).sum())} occurrences).")
print(f"Schéma retenu comme référence : « {PRIMARY_IMPUTATION} » "
      f"(limite de détection propre au tissu, {TISSUE_MIN.min():.2f}–{TISSUE_MIN.max():.2f}).")
print("\nNOTE MATHÉMATIQUE (correction R3.4) : l'hermiticité de H découle de ce que H est")
print("réelle et symétrique, H = (H + H^T)/2. Elle n'exige AUCUNE positivité des éléments.")
print("L'affirmation contraire figurant dans la version précédente était erronée.")
print("La positivité des poids est une contrainte d'INTERPRÉTATION (un couplage négatif")
print("n'aurait pas de sens comme abondance partagée), pas une contrainte algébrique.")


---
## 2. Topologie du réseau $G=(V,E)$

- **Nœuds $V$** : les MitoPathways de la feuille C (≈149).
- **Arêtes $E$** : deux voies sont reliées si elles partagent au moins `MIN_SHARED_GENES` gènes.
- La matrice de membership $M \in \{0,1\}^{n_{genes}\times n_{voies}}$ code l'appartenance gène→voie ;
  tout le reste s'en déduit par algèbre linéaire (aucune boucle Python sur les paires de voies).

In [ ]:
# ============================================================================
# 2.1 Voies -> ensembles de gènes -> matrice de membership M
# ============================================================================
def split_genes(cell):
    if pd.isna(cell):
        return []
    return [g.strip() for g in re.split(r"[,;|]", str(cell)) if g.strip() and g.strip().lower() != "nan"]


def parse_pathways(df_c, path_col=None, genes_col=None):
    path_col = PATH_COL if path_col is None else path_col
    genes_col = GENES_COL if genes_col is None else genes_col
    t = df_c[[path_col, genes_col]].copy()
    t.columns = ["hierarchy", "genes_raw"]
    t["hierarchy"] = t["hierarchy"].astype(str).str.strip()
    t = t[t["hierarchy"].ne("") & t["hierarchy"].ne("nan") & t["genes_raw"].notna()]
    t["leaf"] = t["hierarchy"].str.split(">").str[-1].str.strip()
    t["root"] = t["hierarchy"].str.split(">").str[0].str.strip()
    t["depth"] = t["hierarchy"].str.count(">")
    t["genes"] = t["genes_raw"].map(split_genes)
    return t.reset_index(drop=True)


def build_network(pw_all, keep_leaves_only=CONFIG["KEEP_LEAVES_ONLY"],
                  min_genes=CONFIG["MIN_GENES_PER_PATHWAY"]):
    '''Renvoie (table des voies, matrice de membership M, hiérarchies, noms courts).'''
    t = pw_all.copy()
    if keep_leaves_only:   # une voie est terminale si aucune autre ne la prolonge
        hs = t["hierarchy"].tolist()
        t = t[[not any(h2.startswith(h + " >") for h2 in hs if h2 != h) for h in hs]]
    t["genes_kept"] = t["genes"].map(lambda gs: sorted({g for g in gs if g in GENE_INDEX}))
    t["n_genes"] = t["genes_kept"].map(len)
    t = t[t["n_genes"] >= min_genes].reset_index(drop=True)
    Mx = np.zeros((len(GENES), len(t)))
    for j, gs in enumerate(t["genes_kept"]):
        Mx[[GENE_INDEX[g] for g in gs], j] = 1.0
    return t, Mx, t["hierarchy"].tolist(), t["leaf"].tolist()


PW_ALL = parse_pathways(df_c)
pw, M, PATHWAYS, LEAVES = build_network(PW_ALL)
N = len(PATHWAYS)

print(f"{len(PW_ALL)} voies dans la feuille C  ->  {N} nœuds retenus "
      f"(>= {CONFIG['MIN_GENES_PER_PATHWAY']} gènes quantifiés)")
print(f"{int(M.sum())} relations gène-voie | taille médiane d'une voie : {int(np.median(M.sum(0)))} gènes")
print(f"Couverture : {int((M.sum(1) > 0).sum())}/{len(GENES)} protéines annotées dans au moins une voie")
display(pw.groupby("root")["n_genes"].agg(["count", "median"]).rename(
    columns={"count": "n_voies", "median": "gènes (médiane)"}))

In [ ]:
# ============================================================================
# 2.2 Structure du graphe : chevauchements, degré, connexité
# ============================================================================
SHARED = M.T @ M                                   # |G_ij| : nb de gènes partagés
ADJ = (SHARED >= CONFIG["MIN_SHARED_GENES"]).astype(int)
np.fill_diagonal(ADJ, 0)

n_comp, comp_labels = connected_components(ADJ, directed=False)
degrees = ADJ.sum(1)
net_stats = pd.DataFrame({
    "n_nodes": [N],
    "n_edges": [int(ADJ.sum() // 2)],
    "density": [ADJ.sum() / (N * (N - 1))],
    "mean_degree": [degrees.mean()],
    "n_connected_components": [n_comp],
    "largest_component_size": [int(np.bincount(comp_labels).max())],
    "mean_shared_genes": [SHARED[np.triu_indices(N, 1)][ADJ[np.triu_indices(N, 1)] > 0].mean()],
})
net_stats.to_csv(TABDIR / "network_stats.csv", index=False)
display(net_stats.round(3))

if n_comp > 1:
    print(f"[!] {n_comp} composantes — la marche reste confinée dans celle du nœud source.\n"
          f"    Option : ne garder que la composante géante (voir cellule suivante).")

In [ ]:
# ============================================================================
# 2.3 Redondance hiérarchique — spécifique à MitoCarta (voies emboîtées)
# ============================================================================
# 'OXPHOS' contient tous les gènes de 'OXPHOS > Complex I' : ces arêtes d'inclusion
# ne traduisent pas une proximité fonctionnelle mais la structure de l'annotation.
sizes = M.sum(0)
iu = np.triu_indices(N, 1)
has_edge = SHARED[iu] > 0
containment = SHARED[iu] == np.minimum(sizes[iu[0]], sizes[iu[1]])
jacc = SHARED[iu] / (sizes[iu[0]] + sizes[iu[1]] - SHARED[iu] + 1e-12)

print(f"Arêtes d'inclusion parent-enfant : {100*containment[has_edge].mean():.1f} % des arêtes")
print(f"Indice de Jaccard médian des arêtes : {np.median(jacc[has_edge]):.3f}")
print("→ Deux topologies sont analysées : les 149 voies (référence, cf. plan de l'étude)\n"
      "   et les seules voies terminales (KEEP_LEAVES_ONLY=True), testées en §9.4.\n"
      "   La concordance des deux est l'argument à opposer au reviewer sur ce point.")

In [ ]:
# --- Option : restriction à la composante connexe géante --------------------
RESTRICT_TO_GIANT = True
if RESTRICT_TO_GIANT and n_comp > 1:
    keep = np.where(comp_labels == np.bincount(comp_labels).argmax())[0]
    M, SHARED, ADJ = M[:, keep], SHARED[np.ix_(keep, keep)], ADJ[np.ix_(keep, keep)]
    pw = pw.iloc[keep].reset_index(drop=True)
    PATHWAYS = pw["hierarchy"].tolist(); LEAVES = pw["leaf"].tolist(); N = len(PATHWAYS)
    print(f"Réseau restreint à la composante géante : {N} nœuds")

---
## 3. Construction de l'hamiltonien tissu-spécifique

$$H^{(T)}_{ij} = \frac{1}{|G_{ij}|}\sum_{g\in G_{ij}} I(g,T), \qquad G_{ij}= \text{gènes partagés par } i \text{ et } j$$

Implémentation vectorisée : avec $M$ la matrice de membership et $x$ le vecteur d'intensités,
$$\underbrace{M^{\top}\,\mathrm{diag}(x)\,M}_{\text{numérateur}} \oslash \underbrace{M^{\top}M}_{|G_{ij}|}$$

$H$ est réelle symétrique donc hermitienne, ce qui garantit l'unitarité de $U(t)=e^{-iHt}$.
Normalisation $\lVert H\rVert_2 = 1$ → le temps est adimensionnel et **comparable entre tissus**
(sans cela, un tissu globalement plus abondant « marcherait » simplement plus vite : artefact d'échelle).

In [ ]:
# ============================================================================
# 3.1 Constructeur d'hamiltonien
# ============================================================================
def build_hamiltonian(M, x, include_diagonal=CONFIG["INCLUDE_DIAGONAL"],
                      min_shared=CONFIG["MIN_SHARED_GENES"], normalize=CONFIG["NORMALIZE"]):
    """H_ij = intensité moyenne des gènes partagés par les voies i et j."""
    x = np.asarray(x, float)
    num = M.T @ (M * x[:, None])          # somme des intensités sur les gènes partagés
    cnt = M.T @ M                          # |G_ij|
    with np.errstate(invalid="ignore", divide="ignore"):
        H = np.where(cnt >= min_shared, num / np.maximum(cnt, 1e-12), 0.0)
    if not include_diagonal:
        np.fill_diagonal(H, 0.0)
    H = 0.5 * (H + H.T)                    # symétrisation numérique -> hermiticité exacte
    scale = 1.0
    if normalize == "spectral":
        scale = np.abs(np.linalg.eigvalsh(H)).max()
        H = H / scale if scale > 0 else H
    return H, scale


# Vérifications formelles (à citer dans la section Methods)
H_test, _ = build_hamiltonian(M, X[:, 0])
assert np.allclose(H_test, H_test.conj().T), "H non hermitienne"
U_test = sla.expm(-1j * H_test * 3.7)
assert np.allclose(U_test @ U_test.conj().T, np.eye(N), atol=1e-9), "U non unitaire"
print(f"Hermiticité et unitarité vérifiées (erreur max {np.abs(U_test @ U_test.conj().T - np.eye(N)).max():.2e})")

In [ ]:
# ============================================================================
# 3.2 Hamiltoniens des 14 tissus + spectres
# ============================================================================
HAM, SCALES = {}, {}
for j, t in enumerate(TISSUES):
    HAM[t], SCALES[t] = build_hamiltonian(M, X[:, j])

EIG = {t: np.linalg.eigvalsh(HAM[t]) for t in TISSUES}
spec = pd.DataFrame({
    "tissue": TISSUES,
    "raw_spectral_norm": [SCALES[t] for t in TISSUES],
    "lambda_min": [EIG[t].min() for t in TISSUES],
    "lambda_max": [EIG[t].max() for t in TISSUES],
    "spectral_gap": [np.diff(np.sort(EIG[t]))[-1] for t in TISSUES],
    "spectral_entropy": [(lambda p: -(p * np.log(p + 1e-300)).sum())(
        np.abs(EIG[t]) / np.abs(EIG[t]).sum()) for t in TISSUES],
}).sort_values("raw_spectral_norm", ascending=False)
spec.to_csv(TABDIR / "hamiltonian_spectra.csv", index=False)
display(spec.round(4))

In [ ]:
# --- Figure 1 : densité spectrale des hamiltoniens tissulaires ---------------
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for t in TISSUES:
    axes[0].plot(np.sort(EIG[t]), lw=1, alpha=0.75, label=t)
axes[0].set_xlabel("Eigenvalue index"); axes[0].set_ylabel(r"$\lambda_k$")
axes[0].set_title("Spectra of $H^{(T)}$ (normalised)")
axes[0].legend(ncol=2, fontsize=6)

order = spec["tissue"].tolist()
axes[1].barh(order[::-1], spec["raw_spectral_norm"][::-1], color="0.35")
axes[1].set_xlabel(r"$\|H\|_2$ before normalisation")
axes[1].set_title("Overall coupling magnitude")
fig.suptitle("Tissue-resolved Hamiltonians", y=1.04)
savefig(fig, "fig1_hamiltonian_spectra", "Hamiltonian spectra and coupling norm by tissue.")
plt.show()

---
## 4. Moteur de marche quantique à temps continu

$$\vert\psi(t)\rangle = e^{-iHt}\vert\psi(0)\rangle
= \sum_k e^{-i\lambda_k t}\,\vert v_k\rangle\langle v_k\vert\psi(0)\rangle$$

Une seule diagonalisation $O(n^3)$ par tissu, puis **toutes** les évolutions temporelles s'obtiennent
par produit matriciel — indispensable pour les milliers de réévaluations des tests de permutation.

On implémente aussi le **modèle de piégeage** (non hermitien) :
$H_{\text{eff}} = H - i\Gamma\,\vert s\rangle\langle s\vert$, dont la perte de norme
$\eta(T)=1-\lVert\psi(T)\rVert^2$ mesure l'efficacité de transport vers la voie cible $s$.

In [ ]:
# ============================================================================
# 4.1 Classes de propagation
# ============================================================================
class QuantumWalk:
    """CTQW unitaire : diagonalisation unique, évolutions vectorisées."""

    def __init__(self, H):
        self.H = H
        self.n = H.shape[0]
        self.evals, self.evecs = np.linalg.eigh(H)

    def U(self, t):
        return (self.evecs * np.exp(-1j * self.evals * t)) @ self.evecs.conj().T

    def amplitudes(self, source, times):
        """psi_j(t) pour tous les j et tous les t -> matrice (n_t × n)."""
        c = self.evecs.conj()[source, :]
        phases = np.exp(-1j * np.outer(np.asarray(times, float), self.evals))
        return (phases * c[None, :]) @ self.evecs.T

    def probabilities(self, source, times):
        return np.abs(self.amplitudes(source, times)) ** 2


class TrappingWalk:
    """Marche avec puits absorbant : H_eff = H - i*Gamma*|sink><sink| (non hermitien)."""

    def __init__(self, H, sink, gamma=CONFIG["SINK_RATE"]):
        Heff = H.astype(complex).copy()
        Heff[sink, sink] -= 1j * gamma
        self.w, self.V = sla.eig(-1j * Heff)          # dpsi/dt = -i H_eff psi
        self.Vinv = np.linalg.inv(self.V)

    def norm(self, source, times):
        c = self.Vinv[:, source]
        psi = (np.exp(np.outer(np.asarray(times, float), self.w)) * c[None, :]) @ self.V.T
        return np.sum(np.abs(psi) ** 2, axis=1)

    def efficiency(self, source, times):
        """eta(t) = population absorbée par le puits."""
        return 1.0 - self.norm(source, times)


class ClassicalWalk:
    """Marche aléatoire classique à temps continu : p(t) = e^{-Lt} p(0), L = D - A."""

    def __init__(self, H):
        A = np.abs(H).copy(); np.fill_diagonal(A, 0.0)
        L = np.diag(A.sum(1)) - A
        s = np.abs(np.linalg.eigvalsh(L)).max()
        self.L = L / s if s > 0 else L
        self.evals, self.evecs = np.linalg.eigh(self.L)

    def probabilities(self, source, times):
        c = self.evecs[source, :]
        decay = np.exp(-np.outer(np.asarray(times, float), self.evals))
        p = (decay * c[None, :]) @ self.evecs.T
        return np.clip(p, 0, None)


TIMES = np.linspace(0, CONFIG["T_MAX"], CONFIG["N_T"])
WALKS = {t: QuantumWalk(HAM[t]) for t in TISSUES}
print("Diagonalisation des 14 hamiltoniens terminée.")

In [ ]:
# ============================================================================
# 4.2 Sélection des voies source et cible
# ============================================================================
def find_pathway(query, pathways=None, leaves=None):
    """Recherche par nom de feuille exact, puis par sous-chaîne ; -1 si absent."""
    pathways = PATHWAYS if pathways is None else pathways
    leaves = LEAVES if leaves is None else leaves
    q = query.strip().lower()
    for i, lf in enumerate(leaves):
        if lf.strip().lower() == q:
            return i
    for i, h in enumerate(pathways):
        if q in h.lower():
            return i
    return -1


SOURCE = find_pathway(CONFIG["SOURCE_PATHWAY"])
TARGET = find_pathway(CONFIG["TARGET_PATHWAY"])
if SOURCE < 0 or TARGET < 0:      # repli : nœuds de plus fort et plus faible degré
    deg = ADJ.sum(1)
    SOURCE = int(deg.argmax()) if SOURCE < 0 else SOURCE
    TARGET = int(np.argsort(deg)[-2]) if TARGET < 0 else TARGET
    print("[!] Voies demandées introuvables — repli sur les hubs du réseau.")
print(f"Source : [{SOURCE}] {PATHWAYS[SOURCE]}")
print(f"Cible  : [{TARGET}] {PATHWAYS[TARGET]}")
print(f"Gènes partagés source/cible : {int(SHARED[SOURCE, TARGET])}")

---
## 5. Métriques de performance

| Métrique | Définition | Interprétation |
|---|---|---|
| $P_{i\to j}(t)$ | $\lvert\langle j\rvert e^{-iHt}\lvert i\rangle\rvert^2$ | probabilité de transition |
| QTE | $\frac{1}{T}\int_0^T P_{i\to j}(t)\,dt$ | efficacité de transport moyennée |
| $\eta$ | $1-\lVert\psi(T)\rVert^2$ (puits) | efficacité de piégeage |
| $\tau_{\text{mix}}$ | $\min T : \lVert\bar\chi(T)-u\rVert_1<\varepsilon$ | temps de mélange quantique |
| IPR | $\left(\sum_j p_j^2\right)^{-1}$ | nombre de voies effectivement explorées |
| $C_{\ell_1}$ | $\left(\sum_j\lvert\psi_j\rvert\right)^2-1$ | cohérence dans la base des voies |

La distribution **moyennée en temps** $\bar\chi_j(T)=\frac1T\int_0^T P_j(t)dt$ est utilisée car une marche
quantique unitaire ne converge jamais (pas de distribution stationnaire) — point classique que le reviewer vérifiera.

In [ ]:
# ============================================================================
# 5.1 Fonctions de métriques
# ============================================================================
def qte(walk, source, target, times=TIMES):
    """Efficacité de transport quantique : probabilité d'arrivée moyennée en temps."""
    p = walk.probabilities(source, times)[:, target]
    return TRAPZ(p, times) / (times[-1] - times[0])


def time_averaged_distribution(walk, source, times=TIMES):
    P = walk.probabilities(source, times)
    return TRAPZ(P, times, axis=0) / (times[-1] - times[0])


def mixing_time(walk, source, times=TIMES, eps=0.25):
    """Plus petit T tel que la distance L1 à l'uniforme passe sous eps."""
    P = walk.probabilities(source, times)
    cum = np.cumsum(0.5 * (P[1:] + P[:-1]) * np.diff(times)[:, None], axis=0)
    chi = cum / times[1:, None]
    tv = np.abs(chi - 1.0 / walk.n).sum(1)
    below = np.where(tv < eps)[0]
    return (times[1:][below[0]], tv) if below.size else (np.nan, tv)


def ipr(P):
    """Inverse participation ratio : nb effectif de voies occupées."""
    return 1.0 / np.sum(P ** 2, axis=-1)


def l1_coherence(walk, source, times=TIMES):
    A = np.abs(walk.amplitudes(source, times))
    return A.sum(1) ** 2 - 1.0


def spread_speed(P, times, thresh=1e-3):
    """Nombre de voies atteintes au-dessus du seuil, en fonction du temps."""
    return (P > thresh).sum(1)

In [ ]:
# ============================================================================
# 5.2 Métriques par paire de voies — analyse ILLUSTRATIVE (voir §5.4 pour le critère principal)
# ============================================================================
rows = []
for t in TISSUES:
    qw = WALKS[t]
    P = qw.probabilities(SOURCE, TIMES)
    cw = ClassicalWalk(HAM[t])
    Pc = cw.probabilities(SOURCE, TIMES)
    tw = TrappingWalk(HAM[t], TARGET)
    tau, tv = mixing_time(qw, SOURCE)
    rows.append({
        "tissue": t,
        "QTE_quantum": qte(qw, SOURCE, TARGET),
        "QTE_classical": TRAPZ(Pc[:, TARGET], TIMES) / CONFIG["T_MAX"],
        "P_max_target": P[:, TARGET].max(),
        "t_first_max": TIMES[P[:, TARGET].argmax()],
        "trapping_efficiency": tw.efficiency(SOURCE, [CONFIG["T_MAX"]])[0],
        "mixing_time": tau,                 # NaN = seuil non atteint dans la fenêtre
        "L1_to_uniform_at_Tmax": tv[-1],    # à rapporter à la place quand mixing_time est NaN
        "mean_IPR": ipr(P).mean(),
        "IPR_classical": ipr(Pc).mean(),
        "mean_coherence_l1": l1_coherence(qw, SOURCE).mean(),
        "spectral_gap": np.diff(np.sort(EIG[t]))[-1],
    })
metrics = pd.DataFrame(rows)
metrics["quantum_advantage"] = metrics["QTE_quantum"] / metrics["QTE_classical"].replace(0, np.nan)
metrics = metrics.sort_values("QTE_quantum", ascending=False).reset_index(drop=True)
metrics.to_csv(TABDIR / "tissue_metrics.csv", index=False)
display(metrics.round(4))

n_nan = int(metrics["mixing_time"].isna().sum())
if n_nan:
    print(f"[!] Temps de mélange non atteint (eps=0.25) pour {n_nan}/{len(TISSUES)} tissus dans "
          f"T<{CONFIG['T_MAX']:g}.\n    Ne PAS publier une colonne de NaN : rapporter "
          f"L1_to_uniform_at_Tmax (médiane {metrics['L1_to_uniform_at_Tmax'].median():.3f}) et conclure\n"
          f"    que la marche ne s'équilibre pas — c'est une observation de localisation, pas une donnée manquante.")
adv = metrics["quantum_advantage"]
print(f"\nAvantage quantique : {adv.min():.2f}–{adv.max():.2f} (médiane {adv.median():.2f}). "
      f"{(adv < 1).sum()}/{len(adv)} tissus < 1.")
print("→ La marche classique atteint la cible plus efficacement. À rapporter comme résultat, "
      "pas à dissimuler :\n   c'est la signature attendue d'un régime localisé (cf. §5.4).")


In [ ]:
# --- Figure 2 : dynamique de transfert, tissus à forte vs faible demande énergétique
HIGH = [t for t in ("heart", "skeletalmuscle", "kidney", "brainstem") if t in TISSUES]
LOW = [t for t in ("adipose", "placenta", "stomach", "testis") if t in TISSUES]
if not HIGH or not LOW:                       # repli générique
    HIGH, LOW = TISSUES[:3], TISSUES[-3:]

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.3), sharey=True)
for t in HIGH:
    axes[0].plot(TIMES, WALKS[t].probabilities(SOURCE, TIMES)[:, TARGET], label=t)
for t in LOW:
    axes[1].plot(TIMES, WALKS[t].probabilities(SOURCE, TIMES)[:, TARGET], label=t, ls="--")
for ax, title in zip(axes, ["High energy demand tissues", "Secondary metabolic tissues"]):
    ax.set_xlabel("Dimensionless time $t$"); ax.set_title(title); ax.legend()
axes[0].set_ylabel(r"$P_{\mathrm{source}\to\mathrm{target}}(t)$")
fig.suptitle(f"Quantum propagation: {LEAVES[SOURCE]} to {LEAVES[TARGET]}", y=1.05)
savefig(fig, "fig2_transition_probabilities", "Pathway-to-pathway transition probabilities by tissue.")
plt.show()

In [ ]:
# --- Figure 3 : quantique vs classique (ballistique vs diffusif) -------------
t_ref = HIGH[0]
qw, cw = WALKS[t_ref], ClassicalWalk(HAM[t_ref])
Pq, Pc = qw.probabilities(SOURCE, TIMES), cw.probabilities(SOURCE, TIMES)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
axes[0].plot(TIMES, Pq[:, TARGET], label="quantum (CTQW)")
axes[0].plot(TIMES, Pc[:, TARGET], label="classical (CTRW)")
axes[0].set_ylabel(r"$P_{\to\mathrm{target}}(t)$"); axes[0].legend(); axes[0].set_title("Arrival at target pathway")

axes[1].plot(TIMES, ipr(Pq), label="quantum"); axes[1].plot(TIMES, ipr(Pc), label="classical")
axes[1].set_ylabel("IPR (pathways explored)"); axes[1].legend(); axes[1].set_title("Spreading")

axes[2].plot(TIMES, l1_coherence(qw, SOURCE), color="0.2")
axes[2].set_ylabel(r"$\ell_1$ coherence"); axes[2].set_title("Coherence in the pathway basis")
for ax in axes:
    ax.set_xlabel("Time $t$")
fig.suptitle(f"Quantum versus classical propagation ({t_ref})", y=1.06)
savefig(fig, "fig3_quantum_vs_classical", "Quantum propagation versus classical diffusion on the same network.")
plt.show()

In [ ]:
# --- Figure 4 : temps de mélange et avantage quantique ----------------------
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.2))
for t in TISSUES:
    _, tv = mixing_time(WALKS[t], SOURCE)
    axes[0].plot(TIMES[1:], tv, lw=1, alpha=0.8, label=t)
axes[0].axhline(0.25, color="k", ls=":", lw=1)
axes[0].set_xlabel("Window $T$"); axes[0].set_ylabel(r"$\|\bar\chi(T)-u\|_1$")
axes[0].set_title("Convergence of the time-averaged distribution"); axes[0].legend(ncol=2, fontsize=6)

m = metrics.sort_values("quantum_advantage")
axes[1].barh(m["tissue"], m["quantum_advantage"], color="0.35")
axes[1].axvline(1, color="k", ls=":", lw=1)
axes[1].set_xlabel("Quantum-to-classical transfer-efficiency ratio"); axes[1].set_title("Transfer-efficiency ratio by tissue")
fig.suptitle("Mixing and transfer-efficiency ratio", y=1.05)
savefig(fig, "fig4_mixing_and_advantage", "Quantum mixing time and quantum-to-classical transfer-efficiency ratio.")
plt.show()

### 5.3 Carte de probabilité de transfert moyenne — critère principal recommandé

La QTE d'une **paire unique** dépend fortement du couplage direct entre les deux voies : Complexe I et
Complexe V ne partagent qu'un seul gène dans MitoCarta 3.0, si bien que le transfert emprunte de longs
chemins indirects et que la statistique devient instable (voir §9.4). On calcule donc aussi la
**matrice de transfert moyennée en temps**

$$\chi^{(T)}_{ij} = \frac{1}{T}\int_0^T \left|\langle j\right|e^{-iH_T t}\left|i\rangle\right|^2 dt$$

et sa moyenne hors diagonale — la **probabilité de transfert moyenne** du réseau tissulaire. Ce critère agrège
~9 500 paires de voies : il est bien plus robuste, et c'est celui que je recommande comme critère
principal du manuscrit, la paire Complexe I → Complexe V devenant une illustration mécanistique.

**Identité utile** (à démontrer en Methods) : la conservation de la probabilité impose
$\\sum_j \\chi_{ij}=1$, donc la probabilité de transfert moyenne vaut exactement
$\\bar\\chi_{i\\neq j} = (1-\\overline{\\chi_{ii}})/(n-1)$ : c'est le complément de la
**probabilité de retour moyenne**, mesure standard de localisation quantique. Cela ramène son
évaluation de $O(n^3)$ à $O(n^2)$ par pas de temps et rend les tests de permutation abordables.

In [ ]:
# ============================================================================
# 5.3 Matrices de transfert moyennées en temps et probabilité de transfert moyenne
# ============================================================================
COARSE = np.linspace(0, CONFIG["T_MAX"], 121)


def avg_transfer_matrix(walk, times=COARSE):
    # chi_ij = probabilité de transfert i -> j moyennée sur la fenêtre temporelle
    V, lam = walk.evecs, walk.evals
    acc = np.zeros((walk.n, walk.n))
    for t in times:
        U = (V * np.exp(-1j * lam * t)) @ V.conj().T
        acc += np.abs(U) ** 2
    return acc / len(times)


def mean_return_probability(walk, times=COARSE):
    # moyenne temporelle de chi_ii, calculée en O(n^2) par pas de temps
    V, lam = walk.evecs, walk.evals
    V2 = V ** 2
    acc = 0.0
    for t in times:
        acc += np.abs(V2 @ np.exp(-1j * lam * t)) ** 2
    return float(np.mean(acc / len(times)))


def mean_transfer_probability(walk, times=COARSE, n=None):
    # Identité exacte : sum_j chi_ij = 1 pour tout i, donc la moyenne hors diagonale
    # vaut (1 - moyenne des chi_ii) / (n - 1). Le critère global se ramène ainsi
    # à la probabilité de retour moyenne -> évaluation ~100x plus rapide.
    n = walk.n if n is None else n
    return (1.0 - mean_return_probability(walk, times)) / (n - 1)


CHI = {t: avg_transfer_matrix(WALKS[t]) for t in TISSUES}
off = ~np.eye(N, dtype=bool)

# vérification numérique de l'identité (à citer en Methods)
_chk = (CHI[TISSUES[0]][off].mean(), mean_transfer_probability(WALKS[TISSUES[0]]))
print(f"Identité conductivité / probabilité de retour : {_chk[0]:.6e} vs {_chk[1]:.6e}")
assert np.isclose(*_chk, rtol=1e-6), "identité violée — vérifier la conservation de la probabilité"

glob = pd.DataFrame({
    "tissue": TISSUES,
    "mean_transfer_probability": [CHI[t][off].mean() for t in TISSUES],
    "localization": [np.mean(np.diag(CHI[t])) for t in TISSUES],
    "transfer_entropy": [(lambda c: -(c * np.log(c + 1e-300)).sum(1).mean())(
        CHI[t] / CHI[t].sum(1, keepdims=True)) for t in TISSUES],
}).sort_values("mean_transfer_probability", ascending=False).reset_index(drop=True)
glob.to_csv(TABDIR / "mean_transfer_probability.csv", index=False)
display(glob.round(5))

# Paires de voies les plus discriminantes entre tissu à forte et à faible demande énergétique
d = CHI[HIGH[0]] - CHI[LOW[0]]
iu2 = np.triu_indices(N, 1)
top = np.argsort(np.abs(d[iu2]))[::-1][:20]
pairs = pd.DataFrame({
    "pathway_i": [LEAVES[iu2[0][k]] for k in top],
    "pathway_j": [LEAVES[iu2[1][k]] for k in top],
    "chi_high": [CHI[HIGH[0]][iu2[0][k], iu2[1][k]] for k in top],
    "chi_low": [CHI[LOW[0]][iu2[0][k], iu2[1][k]] for k in top],
    "difference": [d[iu2[0][k], iu2[1][k]] for k in top],
}).rename(columns={"chi_high": f"chi_{HIGH[0]}", "chi_low": f"chi_{LOW[0]}"})
pairs.to_csv(TABDIR / "top_differential_pairs.csv", index=False)
display(pairs.round(4))

In [ ]:
# --- Figure 9 : cartes de conductivité et différentiel tissulaire -----------
ordr = np.argsort([p.split(" > ")[0] for p in PATHWAYS])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, t in zip(axes[:2], [HIGH[0], LOW[0]]):
    im = ax.imshow(np.log10(CHI[t][np.ix_(ordr, ordr)] + 1e-6), cmap="magma")
    ax.set_title(f"$\\log_{{10}}\\chi$ — {t}"); fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xlabel("pathway j"); ax.set_ylabel("pathway i")
v = np.abs(d).max()
im = axes[2].imshow(d[np.ix_(ordr, ordr)], cmap="RdBu_r", vmin=-v, vmax=v)
axes[2].set_title(f"{HIGH[0]} - {LOW[0]}"); fig.colorbar(im, ax=axes[2], fraction=0.046)
axes[2].set_xlabel("pathway j"); axes[2].set_ylabel("pathway i")
fig.suptitle("Mean transfer probability of the network, pathways ordered by category", y=1.04)
savefig(fig, "fig9_conductivity_maps", "Time-averaged transfer matrices and the difference between tissues.")
plt.show()

---
### 5.4 Critère principal : la localisation, en z-score apparié

$$R^{(T)} = \frac{1}{n}\sum_i \bar\chi^{(T)}_{ii}, \qquad
z_T = \frac{R^{(T)} - \mathbb{E}[R^{(T)}_{\text{perm}}]}{\sigma[R^{(T)}_{\text{perm}}]}$$

$R$ est la **probabilité de retour moyenne** : la fraction de la probabilité qui reste au voisinage de
la voie de départ. Elle est liée à la probabilité de transfert moyenne par l'identité du §5.3, mais s'interprète
plus directement comme une mesure de localisation.

Le nul apparié permute les intensités **entre gènes d'un même tissu**. Il conserve donc exactement la
distribution des intensités — y compris la proportion de valeurs imputées au bruit de fond — et ne
détruit que l'association intensité → gène → voie. C'est ce qui rend $z_T$ comparable entre tissus là
où $R^{(T)}$ ne l'est pas.


In [ ]:
# ============================================================================
# 5.4 CRITÈRE PRINCIPAL — localisation quantique, en z-score apparié au tissu
# ============================================================================
# Statistique : R = (1/n) * somme_i chi_ii, probabilité de retour moyenne.
# R élevé  ->  la marche reste confinée au voisinage de sa voie de départ.
#
# NUL APPARIÉ : les intensités sont permutées ENTRE GÈNES À L'INTÉRIEUR D'UN MÊME
# TISSU. La distribution marginale des intensités est donc conservée exactement —
# y compris la proportion de valeurs imputées — et seule l'affectation
# intensité -> gène -> voie est détruite.
#
# FORMULE EXACTE DE LA p-VALEUR EMPIRIQUE (R3.5), bilatérale, avec la correction
# conventionnelle « +1 » au numérateur et au dénominateur :
#
#     p = (1 + #{b : |R_b - moyenne(R_perm)| >= |R_obs - moyenne(R_perm)|}) / (B + 1)
#
# où B est le nombre de permutations. Cette correction garantit p > 0 et rend le
# test exact au sens de Phipson & Smyth (2010). La plus petite p-valeur atteignable
# vaut donc 1/(B+1). Les q-valeurs sont obtenues par Benjamini-Hochberg sur les
# 14 tissus.
FAST_T = COARSE[::3]


def bh(p):
    """Benjamini-Hochberg : q-valeurs à partir d'un vecteur de p-valeurs."""
    p = np.asarray(p, float); m = len(p); o = np.argsort(p); q = np.empty(m)
    q[o] = np.minimum.accumulate((p[o] * m / (np.arange(m) + 1))[::-1])[::-1]
    return np.clip(q, 0, 1)


def localization_zscores(Mx, Xx, include_diagonal=CONFIG["INCLUDE_DIAGONAL"],
                         n_perm=None, times=None, label="", seed=None):
    n_perm = CONFIG["N_PERMUTATIONS"] if n_perm is None else n_perm
    times = FAST_T if times is None else times
    rng = np.random.default_rng(CONFIG["SEED"] if seed is None else seed)
    rows = []
    for j, t in enumerate(TISSUES):
        H, _ = build_hamiltonian(Mx, Xx[:, j], include_diagonal=include_diagonal)
        obs = mean_return_probability(QuantumWalk(H), times)
        null = np.empty(n_perm)
        for k in range(n_perm):
            Hp, _ = build_hamiltonian(Mx, rng.permutation(Xx[:, j]), include_diagonal=include_diagonal)
            null[k] = mean_return_probability(QuantumWalk(Hp), times)
        p = (1 + np.sum(np.abs(null - null.mean()) >= abs(obs - null.mean()))) / (n_perm + 1)
        rows.append({"config": label, "tissue": t, "return_prob": obs, "null_mean": null.mean(),
                     "null_sd": null.std(), "z_localization": (obs - null.mean()) / (null.std() + 1e-12),
                     "p_value": p, "n_permutations": n_perm})
    out = pd.DataFrame(rows)
    out["q_value_BH"] = bh(out["p_value"])
    return out


t0 = time.time()
LOC = localization_zscores(M, X, n_perm=CONFIG["N_PERMUTATIONS"], label="reference")
LOC = LOC.sort_values("z_localization", ascending=False).reset_index(drop=True)
LOC.to_csv(TABDIR / "localization_primary.csv", index=False)
print(f"{CONFIG['N_PERMUTATIONS']} permutations x {len(TISSUES)} tissus en {time.time()-t0:.0f} s")
print(f"Plus petite p-valeur atteignable : 1/(B+1) = {1/(CONFIG['N_PERMUTATIONS']+1):.2e}")
display(LOC.drop(columns=["config", "n_permutations"]).round(5))

sig = LOC[LOC.q_value_BH < 0.05]["tissue"].tolist()
print(f"Tissus plus localisés que leur nul apparié (BH q < 0.05) : {len(sig)}/{len(TISSUES)}")
print(f"  {', '.join(sig) if sig else 'aucun'}")
print(f"Tous les z sont positifs : {bool((LOC.z_localization > 0).all())}")


In [ ]:
# ============================================================================
# 5.5 Tests de confusion — ce que ferait un reviewer en premier
# ============================================================================
# (a) Les métriques BRUTES suivent-elles la profondeur de détection ?
# (b) Le z-score apparié s'en affranchit-il ?
cond_vec = np.array([glob.set_index("tissue").loc[t, "mean_transfer_probability"] for t in TISSUES])
loc_vec = np.array([glob.set_index("tissue").loc[t, "localization"] for t in TISSUES])
z_vec = np.array([LOC.set_index("tissue").loc[t, "z_localization"] for t in TISSUES])
qte_vec = np.array([metrics.set_index("tissue").loc[t, "QTE_quantum"] for t in TISSUES])

conf = pd.DataFrame([
    {"metric": "probabilité de transfert moyenne (brute)", "rho_vs_missing": spearman(MISSING_FRAC, cond_vec)},
    {"metric": "localisation R (brute)", "rho_vs_missing": spearman(MISSING_FRAC, loc_vec)},
    {"metric": "QTE paire (brute)", "rho_vs_missing": spearman(MISSING_FRAC, qte_vec)},
    {"metric": "z de localisation (apparié)", "rho_vs_missing": spearman(MISSING_FRAC, z_vec)},
])
conf["variance_expliquee_%"] = (100 * conf["rho_vs_missing"] ** 2).round(1)
conf.to_csv(TABDIR / "confound_detection_depth.csv", index=False)
display(conf.round(3))
print("Lecture : |rho| proche de 1 = la métrique mesure la couverture MS, pas la biologie.\n"
      "Seules les lignes à |rho| faible peuvent porter une conclusion inter-tissus.\n")

# --- (c) Diagnostic de l'outlier : d'où vient la QTE anormale du rein ? -----
shared_idx = np.where((M[:, SOURCE] > 0) & (M[:, TARGET] > 0))[0]
print(f"Gènes partageant la voie source et la voie cible : {list(GENES[shared_idx])}")
if len(shared_idx):
    diag = pd.DataFrame(X_RAW[shared_idx].T, index=TISSUES, columns=GENES[shared_idx])
    diag["QTE"] = qte_vec
    display(diag.round(4))
    print("Un NaN (protéine non détectée) sur l'unique gène de liaison réduit le poids de l'arête\n"
          "directe au bruit de fond et effondre la QTE du tissu concerné : artefact technique,\n"
          "à signaler explicitement dans le manuscrit plutôt qu'à interpréter.")


---
### 5.6–5.7 D'où vient la localisation ? — relecteur 1 point 3, relecteur 3 point 2

Les deux relecteurs font la même observation décisive : la localisation subsiste quand on
retire les énergies de site, ce qui contredit l'attribution au désordre diagonal. On traite
cette objection par deux analyses qui n'existaient pas dans la version soumise.

**Décomposition.** On sépare $H = D + W$ et on construit une échelle d'hamiltoniens où
l'hétérogénéité diagonale et hors-diagonale sont activées indépendamment. Ajouter une
constante à $D$ ne change pas $|U_{ij}|$ : seule la dispersion compte.

**Test de la signature d'Anderson.** Dans la localisation d'Anderson au sens physique,
l'amplitude du désordre de site contrôle la longueur de localisation de façon monotone.
On pose $H(\lambda) = W + \lambda (D - \langle D\rangle)$ et on mesure $R(\lambda)$. C'est
une prédiction falsifiable, et elle est testée ici directement.


In [ ]:
# ============================================================================
# 5.6 Décomposition des sources de localisation — R1.3 et R3.2
# ============================================================================
# Les relecteurs 1 et 3 soulignent, à juste titre, que la localisation persiste
# lorsqu'on retire les énergies de site : le désordre diagonal ne peut donc pas en
# être la cause principale. On décompose ici la contribution de chaque ingrédient
# en construisant une échelle d'hamiltoniens emboîtés.
#
#   H = D (diagonale, énergies de site) + W (hors-diagonale, couplages)
#
# Noter qu'ajouter une constante à la diagonale ne change pas |U_ij| (phase globale) :
# seule la DISPERSION de la diagonale compte, pas son niveau.
ADJ_BIN = (SHARED >= CONFIG["MIN_SHARED_GENES"]).astype(float)
np.fill_diagonal(ADJ_BIN, 0.0)


def ladder_hamiltonian(x, mode):
    """Construit H en activant ou désactivant l'hétérogénéité diagonale / hors-diagonale."""
    Hfull, _ = build_hamiltonian(M, x, include_diagonal=True, normalize="none")
    W = Hfull.copy(); np.fill_diagonal(W, 0.0)
    dvec = np.diag(Hfull).copy()
    if mode == "topology only":                 # couplages uniformes, diagonale plate
        H = ADJ_BIN * W[W > 0].mean(); d = np.zeros(N)
    elif mode == "off-diagonal disorder":       # vrais couplages, diagonale plate
        H = W; d = np.zeros(N)
    elif mode == "diagonal disorder":           # couplages uniformes, vraies énergies de site
        H = ADJ_BIN * W[W > 0].mean(); d = dvec
    elif mode == "full":                        # les deux
        H = W; d = dvec
    else:
        raise ValueError(mode)
    H = H + np.diag(d)
    H = 0.5 * (H + H.T)
    s = np.abs(np.linalg.eigvalsh(H)).max()
    return H / s if s > 0 else H


MODES = ["topology only", "off-diagonal disorder", "diagonal disorder", "full"]
rows = []
for t in TISSUES:
    j = TISSUES.index(t)
    for mode in MODES:
        rows.append({"tissue": t, "configuration": mode,
                     "R": mean_return_probability(QuantumWalk(ladder_hamiltonian(X[:, j], mode)), FAST_T)})
ladder = pd.DataFrame(rows)
lwide = ladder.pivot(index="tissue", columns="configuration", values="R")[MODES]
lwide.to_csv(TABDIR / "localization_decomposition.csv")
display(lwide.round(4))

base = lwide["topology only"].mean()
print(f"R sous topologie seule (couplages uniformes, diagonale plate) : {base:.4f}")
for mode in MODES[1:]:
    print(f"  + {mode:24s} R = {lwide[mode].mean():.4f}   "
          f"(variation vs topologie : {100*(lwide[mode].mean()-base)/base:+.1f} %)")
print("\nLecture : si « topology only » produit déjà l'essentiel de R, la localisation est")
print("principalement STRUCTURELLE (graphe creux et hiérarchique), et l'abondance protéique")
print("la module sans la créer. C'est la lecture que les relecteurs 1 et 3 appellent de leurs vœux.")


In [ ]:
# ============================================================================
# 5.7 Test de la signature d'Anderson — R3.2
# ============================================================================
# Dans la localisation d'Anderson au sens physique, la longueur de localisation
# décroît de façon MONOTONE quand on augmente l'amplitude du désordre de site à
# couplage fixé. C'est une prédiction falsifiable, et on la teste directement :
#
#     H(lambda) = W + lambda * (D - <D>)
#
# lambda = 0 : aucun désordre diagonal.  lambda = 1 : le désordre observé.
# Si R(lambda) est plat ou non monotone, l'analogie d'Anderson n'est pas soutenue
# par ces données et le terme doit être abandonné.
LAMBDAS = np.array([0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0])


def disorder_scaled_R(x, lam, times=None):
    times = FAST_T if times is None else times
    Hfull, _ = build_hamiltonian(M, x, include_diagonal=True, normalize="none")
    W = Hfull.copy(); np.fill_diagonal(W, 0.0)
    d = np.diag(Hfull).copy()
    H = W + np.diag(lam * (d - d.mean()))
    H = 0.5 * (H + H.T)
    s = np.abs(np.linalg.eigvalsh(H)).max()
    return mean_return_probability(QuantumWalk(H / s if s > 0 else H), times)


and_rows = []
for t in TISSUES:
    j = TISSUES.index(t)
    for lam in LAMBDAS:
        and_rows.append({"tissue": t, "lambda": lam, "R": disorder_scaled_R(X[:, j], lam)})
anderson = pd.DataFrame(and_rows)
awide = anderson.pivot(index="lambda", columns="tissue", values="R")
awide.to_csv(TABDIR / "anderson_disorder_scaling.csv")
display(awide.round(4))

mean_curve = awide.mean(axis=1)
rho_lam = spearman(LAMBDAS, mean_curve.to_numpy())
obs_vs_zero = 100 * (awide.loc[1.0].mean() - awide.loc[0.0].mean()) / awide.loc[0.0].mean()
print(f"Spearman R vs amplitude du désordre : rho = {rho_lam:+.3f}")
print(f"Au désordre RÉELLEMENT observé (lambda = 1), R s'écarte de {obs_vs_zero:+.2f} % "
      f"du cas sans désordre diagonal.")
print(f"À fort désordre (lambda = {LAMBDAS[-1]:g}), R = {awide.loc[LAMBDAS[-1]].mean():.4f} "
      f"contre {awide.loc[0.0].mean():.4f} à lambda = 0.")
print("\nInterprétation : la réponse d'Anderson (R croissant avec le désordre) n'apparaît qu'à")
print("des amplitudes très supérieures à celles observées. Au désordre réel, le système est")
print("dans un régime où la diagonale ne gouverne PAS la localisation. Le terme « localisation")
print("d'Anderson » est donc abandonné dans le manuscrit révisé au profit de « localisation de")
print("la CTQW » / « étalement restreint », qui décrit ce qui est effectivement mesuré.")


In [ ]:
# --- Figure 11 : origine de la localisation et test de la signature d'Anderson
fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.5))

w = 0.2
xpos = np.arange(len(TISSUES))
for i, mode in enumerate(MODES):
    axes[0].bar(xpos + (i - 1.5) * w, lwide.loc[TISSUES, mode], width=w, label=mode)
axes[0].set_xticks(xpos); axes[0].set_xticklabels(TISSUES, rotation=45, ha="right")
axes[0].set_ylabel("Mean return probability $R$")
axes[0].set_ylim(0, max(0.35, lwide.to_numpy().max() * 1.15))
axes[0].legend(fontsize=6, ncol=2)
axes[0].set_title("Sources of localisation")

for t in TISSUES:
    axes[1].semilogx(np.maximum(LAMBDAS, 1e-2), awide[t], lw=0.9, alpha=0.6)
axes[1].semilogx(np.maximum(LAMBDAS, 1e-2), awide.mean(axis=1), lw=2.2, color="k", label="mean")
axes[1].axvline(1.0, color="crimson", ls="--", lw=1.1)
axes[1].text(1.05, axes[1].get_ylim()[1] * 0.98, "observed disorder", color="crimson",
             fontsize=7, va="top")
axes[1].set_xlabel(r"On-site disorder amplitude $\lambda$")
axes[1].set_ylabel("Mean return probability $R$")
axes[1].set_title("Anderson signature test")
axes[1].legend(fontsize=7)

fig.suptitle("Localisation is structural rather than disorder-driven", y=1.04)
savefig(fig, "fig11_localisation_mechanism",
        "Decomposition of localisation into topology, off-diagonal and diagonal contributions, "
        "and response of the return probability to scaled on-site disorder.")
plt.show()


---
### 5.8 Fenêtre d'observation et grille temporelle — relecteur 3, point 6

Spécification complète des quantités moyennées en temps, et vérification que les conclusions
ne dépendent ni de la fenêtre $T$ ni de la résolution de la grille.


In [ ]:
# ============================================================================
# 5.8 Fenêtre d'observation et résolution temporelle — R3.6
# ============================================================================
# Spécification complète, pour reproductibilité :
#   * chi_ij(T) = (1/T) * integrale_0^T |<j|exp(-iHt)|i>|^2 dt
#   * L'intégrale est évaluée NUMÉRIQUEMENT, par moyenne sur une grille de temps
#     régulièrement espacée (règle du rectangle sur des points équidistants), et
#     NON analytiquement. La forme analytique exacte existe
#     (chi_ij = somme_k |v_k(i)|^2 |v_k(j)|^2 pour un spectre non dégénéré, à la
#     limite T -> infini), mais elle décrit la limite infinie, alors que l'on veut
#     ici une fenêtre finie explicite.
#   * Grille de référence : T = 60 (adimensionné), 41 points, pas 1.5.
#   * H est normalisée à ||H||_2 = 1, donc t est en unités de 1/||H||.
WINDOW_GRID = [(15.0, 41), (30.0, 41), (60.0, 41), (60.0, 121), (120.0, 81), (240.0, 161)]

win_rows = []
for Tw, npts in WINDOW_GRID:
    grid = np.linspace(0.0, Tw, npts)
    for t in TISSUES:
        win_rows.append({"T": Tw, "n_points": npts, "dt": Tw / (npts - 1), "tissue": t,
                         "R": mean_return_probability(WALKS[t], grid)})
window = pd.DataFrame(win_rows)
window["window"] = window["T"].map(lambda v: f"T={v:g}") + window["n_points"].map(lambda v: f", n={v}")
wwide = window.pivot(index="tissue", columns="window", values="R")
wwide.to_csv(TABDIR / "observation_window_stability.csv")
display(wwide.round(4))

ref_col = "T=60, n=41"
print("Concordance des classements tissulaires (Spearman vs la fenêtre de référence) :")
for c in wwide.columns:
    print(f"  {c:18s} rho = {spearman(wwide[ref_col], wwide[c]):+.3f}")

# Convergence de la grille à T fixé
conv = wwide["T=60, n=121"] - wwide["T=60, n=41"]
print(f"\nRaffinement de la grille à T = 60 (41 -> 121 points) : variation max de R = "
      f"{np.abs(conv).max():.5f} ({100*np.abs(conv/wwide[ref_col]).max():.2f} %).")
print("La grille de référence est donc convergée ; les conclusions ne dépendent pas de sa résolution.")


---
## 6. Déphasage et transport — contrôle négatif

Objection classique : *« à 310 K, la décohérence détruit tout »*. On y répond avec une équation maîtresse
de Lindblad à déphasage pur (Haken–Strobl) plus un puits absorbant :

$$\dot\rho = -i[H,\rho] + \gamma\sum_n\left(P_n\rho P_n - \tfrac12\{P_n^2,\rho\}\right)
- \tfrac12\{\Gamma_{\text{sink}},\rho\}$$

Un maximum d'efficacité à $\gamma$ intermédiaire signerait un transport assisté par l'environnement (ENAQT).

**Ce n'est pas ce que donnent ces données** : $\eta(\gamma)$ décroît de façon monotone, l'optimum est à la
borne cohérente. À rapporter comme **résultat négatif** — cohérent avec le régime localisé du §5.4, où le
transport est limité par le piégeage et non par le désordre à franchir. Ne pas conserver de section
« ENAQT » dans les Results ; une phrase en Discussion suffit.


In [ ]:
# ============================================================================
# 6.1 Équation maîtresse de Lindblad (déphasage + puits), intégration RK4
# ============================================================================
def lindblad_efficiency(H, source, sink, gamma_deph, gamma_sink=CONFIG["SINK_RATE"],
                        T=30.0, dt=0.02):
    """Renvoie eta = population absorbée par le puits après un temps T."""
    n = H.shape[0]
    Hc = H.astype(complex)
    G = np.zeros(n); G[sink] = gamma_sink

    def rhs(rho):
        comm = -1j * (Hc @ rho - rho @ Hc)
        deph = -gamma_deph * (rho - np.diag(np.diag(rho)))
        anti = -0.5 * (G[:, None] * rho + rho * G[None, :])
        return comm + deph + anti

    rho = np.zeros((n, n), complex); rho[source, source] = 1.0
    for _ in range(int(T / dt)):
        k1 = rhs(rho); k2 = rhs(rho + 0.5 * dt * k1)
        k3 = rhs(rho + 0.5 * dt * k2); k4 = rhs(rho + dt * k3)
        rho = rho + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return float(np.clip(1.0 - np.real(np.trace(rho)), 0, 1))


def subnetwork(indices):
    """Extrait un sous-réseau (renormalisé) et remappe les indices."""
    idx = np.asarray(sorted(set(indices)))
    remap = {int(v): k for k, v in enumerate(idx)}
    return idx, remap


# Sous-réseau OXPHOS (ou 25 nœuds de plus fort degré) pour un coût de calcul raisonnable
oxphos = [i for i, h in enumerate(PATHWAYS) if "oxphos" in h.lower()]
if len(oxphos) < 6:
    oxphos = list(np.argsort(ADJ.sum(1))[-25:])
oxphos = sorted(set(oxphos) | {SOURCE, TARGET})
IDX, REMAP = subnetwork(oxphos)
print(f"Sous-réseau ENAQT : {len(IDX)} voies")

In [ ]:
# --- Figure 5 : courbe ENAQT eta(gamma) -------------------------------------
GAMMAS = np.logspace(-3, 1.2, 12)
enaqt = {}
t0 = time.time()
for t in (HIGH[:1] + LOW[:1]):
    Hs = HAM[t][np.ix_(IDX, IDX)]
    s = np.abs(np.linalg.eigvalsh(Hs)).max(); Hs = Hs / (s if s > 0 else 1)
    enaqt[t] = [lindblad_efficiency(Hs, REMAP[SOURCE], REMAP[TARGET], g) for g in GAMMAS]
print(f"ENAQT calculé en {time.time()-t0:.1f} s")

fig, ax = plt.subplots(figsize=(5.2, 3.2))
for t, ys in enaqt.items():
    ax.semilogx(GAMMAS, ys, marker="o", ms=3.5, label=t)
    ax.axvline(GAMMAS[int(np.argmax(ys))], ls=":", lw=0.8, color="0.6")
ax.set_xlabel(r"Dephasing rate $\gamma$"); ax.set_ylabel(r"Trapping efficiency $\eta$")
ax.set_title("Dephasing and transfer efficiency")
ax.legend()
savefig(fig, "fig5_enaqt", "Trapping efficiency as a function of the dephasing rate.")
plt.show()

pd.DataFrame({"gamma": GAMMAS, **enaqt}).to_csv(TABDIR / "enaqt_curve.csv", index=False)

---
### 6.2 Paramètres numériques et convergence de Lindblad — relecteur 3, point 10

Le relecteur relève à juste titre qu'un résultat négatif ne vaut que si l'on peut exclure
l'artefact numérique. Tous les paramètres sont donc explicités, et la convergence est
vérifiée en pas de temps et en durée d'intégration.


In [ ]:
# ============================================================================
# 6.2 Paramètres complets et contrôles de convergence de Lindblad — R3.10
# ============================================================================
# Spécification numérique intégrale du calcul ENAQT, telle que demandée :
#
#   Équation      : d(rho)/dt = -i[H,rho] + gamma*(sum_n P_n rho P_n - rho)
#                               - (1/2){Gamma_sink, rho}
#                   (déphasage pur de Haken-Strobl ; le terme de déphasage se réduit
#                   à -gamma * (rho - diag(rho)) car les P_n sont les projecteurs de
#                   la base des voies)
#   État initial  : rho(0) = |source><source|, population 1 sur la voie de départ
#   Puits         : Gamma_sink = gamma_sink * |target><target|, gamma_sink = 1.0
#   Efficacité    : eta(T) = 1 - Tr[rho(T)], population absorbée par le puits
#   Intégrateur   : Runge-Kutta explicite d'ordre 4, pas FIXE
#   Sous-réseau   : voies OXPHOS, renormalisées à ||H||_2 = 1
LIND = dict(gamma_sink=CONFIG["SINK_RATE"], dt=0.02, T=30.0,
            gamma_min=1e-3, gamma_max=16.0, n_gamma=12, spacing="logarithmique")
print("Paramètres de référence :", LIND)
print(f"Sous-réseau : {len(IDX)} voies | source = {LEAVES[SOURCE]} | puits = {LEAVES[TARGET]}")
print(f"Nombre de pas RK4 par courbe : {int(LIND['T']/LIND['dt'])}")

# --- Convergence en pas de temps --------------------------------------------
Hs_ref = HAM[HIGH[0]][np.ix_(IDX, IDX)]
_s = np.abs(np.linalg.eigvalsh(Hs_ref)).max()
Hs_ref = Hs_ref / (_s if _s > 0 else 1.0)

conv_rows = []
for dt in [0.08, 0.04, 0.02, 0.01, 0.005]:
    for g in [1e-3, 0.1, 1.0]:
        conv_rows.append({"dt": dt, "gamma": g,
                          "eta": lindblad_efficiency(Hs_ref, REMAP[SOURCE], REMAP[TARGET], g,
                                                     T=LIND["T"], dt=dt)})
conv = pd.DataFrame(conv_rows).pivot(index="dt", columns="gamma", values="eta")
conv.to_csv(TABDIR / "lindblad_dt_convergence.csv")
display(conv.round(6))
drift = np.abs(conv.loc[0.02] - conv.loc[0.005]).max()
print(f"Écart maximal entre dt = 0.02 (référence) et dt = 0.005 : {drift:.2e}")
print(f"-> le pas de référence est convergé à {drift:.1e} près ; le résultat ENAQT négatif "
      f"n'est pas un artefact d'intégration.")

# --- Convergence en temps total ---------------------------------------------
tot_rows = []
for Tt in [10.0, 20.0, 30.0, 60.0, 120.0]:
    for g in [1e-3, 0.1, 1.0]:
        tot_rows.append({"T_total": Tt, "gamma": g,
                         "eta": lindblad_efficiency(Hs_ref, REMAP[SOURCE], REMAP[TARGET], g,
                                                    T=Tt, dt=LIND["dt"])})
tot = pd.DataFrame(tot_rows).pivot(index="T_total", columns="gamma", values="eta")
tot.to_csv(TABDIR / "lindblad_T_convergence.csv")
display(tot.round(4))
print("Noter que eta croît avec T puisque le puits continue d'absorber : la comparaison entre")
print("valeurs de gamma doit donc se faire à T fixé, ce qui est le cas ici (T = 30 partout).")
print(f"L'ordre des gamma est identique à T = 30 et à T = 120 : "
      f"{bool(np.all(np.argsort(tot.loc[30.0].values) == np.argsort(tot.loc[120.0].values)))}")


---
## 7. Analyse de résilience — inhibition ciblée de gènes

Simulation *in silico* d'une mutation perte-de-fonction : l'intensité du gène est ramenée au bruit de fond,
$H$ est reconstruit, et l'on mesure la dégradation relative du transport
$\Delta\text{QTE}\,(\%) = 100\,(\text{QTE}_{\text{KO}}-\text{QTE}_{\text{WT}})/\text{QTE}_{\text{WT}}$.

C'est le résultat le plus « traduisible » cliniquement : les gènes dont l'inhibition dégrade le plus le
transport sont candidats bio-marqueurs. **À valider** contre une liste externe de gènes de maladies
mitochondriales (MitoMap, OMIM) — c'est ce qui transformera la corrélation en argument.

In [ ]:
# ============================================================================
# 7.1 Knock-out in silico
# ============================================================================
def knockout(genes_ko, tissue_idx, source=None, target=None):
    """Recalcule QTE après mise au bruit de fond des gènes indiqués."""
    source = SOURCE if source is None else source
    target = TARGET if target is None else target
    x = X[:, tissue_idx].copy()
    hit = [GENE_INDEX[g] for g in genes_ko if g in GENE_INDEX]
    if not hit:
        return np.nan, 0
    x[hit] = CONFIG["BACKGROUND"]
    Hko, _ = build_hamiltonian(M, x)
    return qte(QuantumWalk(Hko), source, target), len(hit)


# Cibles : gènes OXPHOS classiques ; repli sur les gènes les plus connectés du réseau
CANDIDATES = ["NDUFS1", "NDUFV1", "NDUFA9", "SDHA", "SDHB", "UQCRC1", "UQCRFS1",
              "MT-CO1", "COX15", "ATP5F1A", "ATP5F1B", "TIMM23", "TOMM40", "OPA1", "MFN2"]
present = [g for g in CANDIDATES if g in GENE_INDEX]
if len(present) < 5:
    present = list(GENES[np.argsort(M.sum(1) * X.mean(1))[-12:]])
    print("[i] Gènes OXPHOS absents (jeu synthétique ?) — repli sur les gènes les plus centraux.")
print(f"{len(present)} gènes testés : {', '.join(present[:15])}")

wt = {t: qte(WALKS[t], SOURCE, TARGET) for t in TISSUES}
ko_rows = []
for g in present:
    for j, t in enumerate(TISSUES):
        val, n_hit = knockout([g], j)
        ko_rows.append({"gene": g, "tissue": t, "QTE_wt": wt[t], "QTE_ko": val,
                        "delta_pct": 100 * (val - wt[t]) / wt[t] if wt[t] else np.nan})
ko = pd.DataFrame(ko_rows)
ko.to_csv(TABDIR / "knockouts.csv", index=False)

ko_summary = (ko.groupby("gene")["delta_pct"].agg(["mean", "std", "min", "max"])
                .sort_values("mean").round(2))
display(ko_summary)

In [ ]:
# --- Figure 6 : carte de vulnérabilité gène × tissu -------------------------
piv = ko.pivot(index="gene", columns="tissue", values="delta_pct").loc[ko_summary.index]
fig, ax = plt.subplots(figsize=(0.42 * len(TISSUES) + 3.2, 0.32 * len(piv) + 2.2))
vmax = np.nanmax(np.abs(piv.to_numpy()))
im = ax.imshow(piv.to_numpy(), cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=45, ha="right")
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
fig.colorbar(im, ax=ax, label=r"$\Delta$ transfer efficiency (%)")
ax.set_title("Impact of single-gene in silico knockouts")
savefig(fig, "fig6_resilience_knockouts", "Change in propagation after targeted in silico knockout, by tissue.")
plt.show()

---
### 7.2 Validation externe — le test qui décide de la publiabilité

Le criblage précédent est descriptif. Ce qui convaincra un reviewer biologiste, c'est que les protéines
dont la perte perturbe le plus le réseau soient **enrichies en gènes de maladie mitochondriale connue**.
On croise donc le criblage systématique avec une liste de gènes OMIM/MitoMap (test de Mann-Whitney sur
l'amplitude d'effet, test exact de Fisher sur le décile supérieur).

Noter que l'on teste **l'amplitude** $|\Delta|$ et non l'effet signé : un knock-out qui *augmente* la
délocalisation reste une perturbation du réseau. C'est aussi la réponse à l'anomalie de la v1, où
l'inhibition d'OPA1 « améliorait » le transport de 15 % — un artefact de reroutage de chemins, pas un
bénéfice physiologique.


In [ ]:
# ============================================================================
# 7.2 Criblage systématique des knock-outs + validation externe (gènes de maladie)
# ============================================================================
from scipy.stats import mannwhitneyu, fisher_exact

# Liste de travail de gènes de maladies mitochondriales mendéliennes (OMIM/MitoMap,
# littérature établie). ⚠️ POUR PUBLICATION : remplacer par le téléchargement daté
# de MitoMap (https://www.mitomap.org) ou la requête OMIM correspondante, et citer
# la version. Cette liste sert à rendre le test exécutable, pas à faire autorité.
DISEASE_GENES = set("""
NDUFS1 NDUFS2 NDUFS3 NDUFS4 NDUFS6 NDUFS7 NDUFS8 NDUFV1 NDUFV2 NDUFA1 NDUFA2 NDUFA9 NDUFA10 NDUFA11
NDUFA12 NDUFAF1 NDUFAF2 NDUFAF3 NDUFAF4 NDUFAF5 NDUFAF6 NDUFB3 NDUFB9 NDUFB11 TIMMDC1 TMEM126B NUBPL
FOXRED1 ACAD9 SDHA SDHB SDHC SDHD SDHAF1 SDHAF2 UQCRB UQCRQ UQCRC2 CYC1 BCS1L TTC19 LYRM7 UQCC2 UQCC3
COX10 COX14 COX15 COX20 COX6B1 COX7B COX8A SURF1 SCO1 SCO2 PET100 PET117 COA3 COA5 COA6 COA7 COA8
TACO1 LRPPRC FASTKD2 ATP5F1A ATP5F1D ATP5F1E ATPAF2 TMEM70 USMG5 MT-ATP6 MT-ND1 MT-ND5 MT-CO1 MT-CO3
MT-CYB MT-TL1 POLG POLG2 TWNK TFAM SSBP1 DGUOK TK2 RRM2B MPV17 SUCLA2 SUCLG1 TYMP ABAT SLC25A4 SLC25A3
SLC25A46 SLC25A12 SLC25A19 SLC25A22 OPA1 MFN2 DNM1L MFF GDAP1 STAT2 AGK TAZ SERAC1 PNPT1 ELAC2 HSD17B10
MTO1 GTPBP3 TRMU PUS1 TRIT1 TRMT5 MTFMT AARS2 CARS2 DARS2 EARS2 FARS2 HARS2 IARS2 KARS1 LARS2 MARS2
NARS2 PARS2 RARS2 SARS2 TARS2 VARS2 WARS2 YARS2 QRSL1 GATB GATC MRPL3 MRPL12 MRPL44 MRPS2 MRPS7 MRPS16
MRPS22 MRPS23 MRPS34 MTRFR GFM1 GFM2 TSFM TUFM RMND1 C1QBP ERAL1 MRM2 FBXL4 SPG7 AFG3L2 PMPCA PMPCB
HTRA2 CLPB CLPP LONP1 PITRM1 XPNPEP3 TIMM8A TIMM22 TIMM50 DNAJC19 GFER PDSS1 PDSS2 COQ2 COQ4 COQ6 COQ7
COQ8A COQ9 ETFA ETFB ETFDH ISCU NFS1 NFU1 BOLA3 IBA57 LIPT1 LIPT2 LIAS GLRX5 FDX2 FDXR FXN ABCB7 ATAD3A
HCCS OXA1L ECHS1 HIBCH PDHA1 PDHB DLD PDHX PDP1 ACO2 FH IDH2 OGDH MDH2 CS GOT2 ALDH18A1 QIL1 CHCHD10
""".split())

# --- criblage : chaque protéine du réseau, dans 3 tissus représentatifs ------
SCAN_TISSUES = [t for t in [HIGH[0], "kidney", LOW[0]] if t in TISSUES]
in_network = np.where(M.sum(1) > 0)[0]
print(f"Criblage de {len(in_network)} protéines x {len(SCAN_TISSUES)} tissus…")

t0 = time.time()
scan = []
for t in SCAN_TISSUES:
    j = TISSUES.index(t)
    R0 = mean_return_probability(WALKS[t], FAST_T)
    q0 = qte(WALKS[t], SOURCE, TARGET)
    for gi in in_network:
        x = X[:, j].copy(); x[gi] = CONFIG["BACKGROUND"]
        Hk, _ = build_hamiltonian(M, x)
        wk = QuantumWalk(Hk)
        scan.append({"gene": GENES[gi], "tissue": t,
                     "delta_localization_pct": 100 * (mean_return_probability(wk, FAST_T) - R0) / R0,
                     "delta_QTE_pct": 100 * (qte(wk, SOURCE, TARGET) - q0) / q0 if q0 else np.nan,
                     "n_pathways": int(M[gi].sum())})
scan = pd.DataFrame(scan)
scan["is_disease_gene"] = scan["gene"].isin(DISEASE_GENES)
scan["abs_effect"] = scan["delta_localization_pct"].abs()
scan.to_csv(TABDIR / "knockout_scan.csv", index=False)
print(f"terminé en {time.time()-t0:.0f} s | {scan.is_disease_gene.sum()//len(SCAN_TISSUES)} "
      f"gènes de maladie présents dans le réseau")

# --- validation externe : les gènes de maladie ont-ils un effet plus fort ? --
val = []
for t in SCAN_TISSUES:
    s = scan[scan.tissue == t]
    a, b = s.loc[s.is_disease_gene, "abs_effect"], s.loc[~s.is_disease_gene, "abs_effect"]
    U, pmw = mannwhitneyu(a, b, alternative="greater")
    thr = s["abs_effect"].quantile(0.90)                    # décile supérieur d'impact
    tab = [[int(((s.abs_effect >= thr) & s.is_disease_gene).sum()),
            int(((s.abs_effect >= thr) & ~s.is_disease_gene).sum())],
           [int(((s.abs_effect < thr) & s.is_disease_gene).sum()),
            int(((s.abs_effect < thr) & ~s.is_disease_gene).sum())]]
    OR, pf = fisher_exact(tab, alternative="greater")
    val.append({"tissue": t, "median_effect_disease": a.median(), "median_effect_other": b.median(),
                "MannWhitney_p": pmw, "odds_ratio_top_decile": OR, "Fisher_p": pf})
validation = pd.DataFrame(val)
validation.to_csv(TABDIR / "disease_gene_validation.csv", index=False)
display(validation.round(4))
print("C'est LE test qui transforme l'exercice mathématique en résultat biologique :\n"
      "un OR > 1 avec p < 0.05 signifie que les protéines dont la perte désorganise le plus\n"
      "le transport sont enrichies en gènes de pathologie mitochondriale connue.")


---
### 7.3–7.4 Enrichissement ajusté sur l'exposition structurale — relecteur 1 point 5, relecteur 3 points 8 et 9

Objection partagée : une protéine présente dans de nombreuses voies perturbe mécaniquement
davantage le réseau. L'enrichissement brut pourrait donc ne mesurer que l'exposition
structurale. On l'ajuste par régression logistique sur trois covariables, et on le re-teste
par permutation des étiquettes de maladie **à l'intérieur de strates d'exposition comparable**.
Le statut exploratoire des tests est énoncé explicitement et la correction de multiplicité
est appliquée.


In [ ]:
# ============================================================================
# 7.3 Enrichissement ajusté sur la position dans le réseau — R1.5, R3.8, R3.9
# ============================================================================
# Objection commune aux relecteurs 1 et 3 : une protéine appartenant à de nombreuses
# voies perturbe mécaniquement davantage le réseau quand on la retire. L'enrichissement
# brut pourrait donc ne refléter que l'exposition structurale, et non le statut
# pathologique. On l'ajuste par trois covariables d'exposition et on le re-teste.
import statsmodels.api as sm

# --- covariables d'exposition structurale -----------------------------------
PATH_DEGREE = ADJ.sum(1)                                   # degré de chaque voie
EXPOSURE = pd.DataFrame({
    "gene": GENES[in_network],
    "n_pathways": M[in_network].sum(1),                    # nb de voies contenant la protéine
    "pathway_degree": M[in_network] @ PATH_DEGREE,         # somme des degrés de ses voies
    "mean_abundance": X[in_network].mean(1),               # abondance moyenne (14 tissus)
})

scan_adj = scan.merge(EXPOSURE, on="gene", how="left", suffixes=("", "_exp"))
scan_adj["n_pathways"] = scan_adj["n_pathways_exp"].fillna(scan_adj["n_pathways"]) \
    if "n_pathways_exp" in scan_adj else scan_adj["n_pathways"]
scan_adj = scan_adj.loc[:, ~scan_adj.columns.str.endswith("_exp")]
scan_adj.to_csv(TABDIR / "knockout_scan_with_exposure.csv", index=False)

print("Corrélation entre amplitude de perturbation et exposition structurale :")
for t in SCAN_TISSUES:
    s = scan_adj[scan_adj.tissue == t]
    print(f"  {t:10s} |delta| vs n_pathways   rho = {spearman(s.abs_effect, s.n_pathways):+.3f}"
          f"   | vs pathway_degree rho = {spearman(s.abs_effect, s.pathway_degree):+.3f}"
          f"   | vs abondance rho = {spearman(s.abs_effect, s.mean_abundance):+.3f}")
print("-> Si ces corrélations sont fortes, l'enrichissement BRUT ne peut pas être interprété seul.")


In [ ]:
# ============================================================================
# 7.4 Régression logistique ajustée et permutation stratifiée — R1.5, R3.8, R3.9
# ============================================================================
# Modèle : statut pathologique ~ |delta| + n_voies + degré des voies + abondance
# Toutes les covariables sont standardisées, donc les OR s'interprètent par écart-type.
def zsc(v):
    v = np.asarray(v, float)
    return (v - v.mean()) / (v.std() + 1e-12)


adj_rows, perm_rows = [], []
for t in SCAN_TISSUES:
    s = scan_adj[scan_adj.tissue == t].dropna(subset=["abs_effect", "n_pathways"]).copy()
    y = s["is_disease_gene"].astype(int).to_numpy()

    # --- (a) modèle brut : |delta| seul --------------------------------------
    Xu = sm.add_constant(zsc(s["abs_effect"]))
    mu = sm.Logit(y, Xu).fit(disp=0)
    # --- (b) modèle ajusté ---------------------------------------------------
    Xa = sm.add_constant(np.column_stack([zsc(s["abs_effect"]), zsc(s["n_pathways"]),
                                          zsc(s["pathway_degree"]), zsc(s["mean_abundance"])]))
    ma = sm.Logit(y, Xa).fit(disp=0)
    ci = ma.conf_int()[1]
    adj_rows.append({
        "tissue": t,
        "OR_unadjusted": float(np.exp(mu.params[1])), "p_unadjusted": float(mu.pvalues[1]),
        "OR_adjusted": float(np.exp(ma.params[1])),
        "CI95_low": float(np.exp(ci[0])), "CI95_high": float(np.exp(ci[1])),
        "p_adjusted": float(ma.pvalues[1]),
        "OR_n_pathways": float(np.exp(ma.params[2])),
        "OR_pathway_degree": float(np.exp(ma.params[3])),
        "OR_abundance": float(np.exp(ma.params[4])),
    })

    # --- (c) permutation stratifiée sur les quintiles de n_voies -------------
    # On permute les ÉTIQUETTES de maladie à l'intérieur de strates d'exposition
    # comparable : le nul conserve donc exactement la distribution d'exposition.
    strata = pd.qcut(s["n_pathways"], q=min(5, s["n_pathways"].nunique()),
                     labels=False, duplicates="drop").to_numpy()
    eff = s["abs_effect"].to_numpy()
    obs = eff[y == 1].mean() - eff[y == 0].mean()
    rng = np.random.default_rng(CONFIG["SEED"])
    null = np.empty(CONFIG["N_PERM_STRATIFIED"])
    for k in range(CONFIG["N_PERM_STRATIFIED"]):
        yp = y.copy()
        for g in np.unique(strata):
            idx = np.where(strata == g)[0]
            yp[idx] = rng.permutation(y[idx])
        null[k] = eff[yp == 1].mean() - eff[yp == 0].mean()
    p_strat = (1 + np.sum(null >= obs)) / (CONFIG["N_PERM_STRATIFIED"] + 1)
    perm_rows.append({"tissue": t, "obs_difference": obs, "null_mean": null.mean(),
                      "null_sd": null.std(), "z": (obs - null.mean()) / (null.std() + 1e-12),
                      "p_stratified": p_strat})

adjusted = pd.DataFrame(adj_rows)
stratified = pd.DataFrame(perm_rows)

# --- correction de multiplicité sur les trois tissus (R3.9) -----------------
def bh(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p); q = np.empty(m)
    q[o] = np.minimum.accumulate((p[o] * m / (np.arange(m) + 1))[::-1])[::-1]
    return np.clip(q, 0, 1)


adjusted["q_adjusted_BH"] = bh(adjusted["p_adjusted"])
stratified["q_stratified_BH"] = bh(stratified["p_stratified"])
adjusted.to_csv(TABDIR / "enrichment_adjusted_logistic.csv", index=False)
stratified.to_csv(TABDIR / "enrichment_stratified_permutation.csv", index=False)

display(adjusted.round(4))
display(stratified.round(4))
print("Ces tests sont EXPLORATOIRES et désignés comme tels dans le manuscrit : les trois tissus")
print("ont été choisis a priori comme représentatifs, sans hypothèse préenregistrée. La correction")
print("de Benjamini-Hochberg est néanmoins appliquée sur les trois tissus et rapportée (colonnes q).")


---
## 8. Clustering tissulaire par signatures quantiques

Distance entre opérateurs d'évolution :
$$d_F(T_1,T_2) = \lVert U_{T_1}(t^\*) - U_{T_2}(t^\*)\rVert_F$$

⚠️ $d_F$ dépend de la phase globale et du choix de $t^\*$. On calcule donc **aussi** une variante
invariante de phase sur les matrices de probabilité $\lvert U_{ij}\rvert^2$, et on vérifie que les deux
donnent la même structure de clusters (anticipation directe d'une critique de reviewer).

In [ ]:
# ============================================================================
# 8.1 Matrices de distance quantique
# ============================================================================
TSTAR = CONFIG["T_STAR"]
U = {t: WALKS[t].U(TSTAR) for t in TISSUES}
Pmat = {t: np.abs(U[t]) ** 2 for t in TISSUES}

def dist_matrix(mats, norm=True):
    ts = list(mats)
    D = np.zeros((len(ts), len(ts)))
    for a, b in itertools.combinations(range(len(ts)), 2):
        D[a, b] = D[b, a] = np.linalg.norm(mats[ts[a]] - mats[ts[b]], "fro")
    return D / D.max() if norm and D.max() > 0 else D

D_U = dist_matrix(U)            # distance de Frobenius sur U(t*)
D_P = dist_matrix(Pmat)         # variante invariante de phase
D_H = dist_matrix(HAM)          # référence statique (hamiltoniens)
D_expr = dist_matrix({t: X[:, j][:, None] for j, t in enumerate(TISSUES)})   # profils d'expression bruts

rho_UP = spearman(squareform(D_U, checks=False), squareform(D_P, checks=False))
rho_UE = spearman(squareform(D_U, checks=False), squareform(D_expr, checks=False))
print(f"Concordance U vs |U|^2 : rho = {rho_UP:.3f}  (doit être élevée -> robustesse au choix de phase)")
print(f"Concordance U vs expression brute : rho = {rho_UE:.3f}  "
      f"(si ~1, la dynamique quantique n'apporte rien de plus que l'expression : à discuter honnêtement)")

pd.DataFrame(D_U, index=TISSUES, columns=TISSUES).to_csv(TABDIR / "tissue_distance_U.csv")
pd.DataFrame(D_P, index=TISSUES, columns=TISSUES).to_csv(TABDIR / "tissue_distance_P.csv")

In [ ]:
# --- Figure 7 : dendrogramme + matrice de distance --------------------------
Z = linkage(squareform(D_U, checks=False), method="average")
clusters = fcluster(Z, t=3, criterion="maxclust")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8),
                         gridspec_kw={"width_ratios": [1.15, 1]})
dn = dendrogram(Z, labels=TISSUES, ax=axes[0], color_threshold=0.7 * Z[:, 2].max(),
                leaf_font_size=7)
axes[0].set_ylabel(r"normalised $d_F$"); axes[0].set_title("Clustering by quantum signature")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")

order = dn["leaves"]
im = axes[1].imshow(D_U[np.ix_(order, order)], cmap="viridis")
axes[1].set_xticks(range(len(TISSUES))); axes[1].set_xticklabels([TISSUES[i] for i in order], rotation=45, ha="right")
axes[1].set_yticks(range(len(TISSUES))); axes[1].set_yticklabels([TISSUES[i] for i in order])
fig.colorbar(im, ax=axes[1], label=r"$\|U_{T_1}(t^*)-U_{T_2}(t^*)\|_F$")
axes[1].set_title(f"Operator distance ($t^*={TSTAR:g}$)")
fig.suptitle("Tissue clustering by quantum signature", y=1.04)
savefig(fig, "fig7_tissue_clustering", "Hierarchical clustering of tissues by distance between evolution operators.")
plt.show()

pd.DataFrame({"tissue": TISSUES, "cluster": clusters}).to_csv(TABDIR / "tissue_clusters.csv", index=False)
display(pd.DataFrame({"tissue": TISSUES, "cluster": clusters}).sort_values("cluster"))

In [ ]:
# ============================================================================
# 8.2 Stabilité du clustering vis-à-vis de t* (analyse de sensibilité)
# ============================================================================
def clustering_at(tstar, k=3):
    Uk = {t: WALKS[t].U(tstar) for t in TISSUES}
    Dk = dist_matrix(Uk)
    return fcluster(linkage(squareform(Dk, checks=False), "average"), k, "maxclust"), Dk


def ari(a, b):
    """Adjusted Rand Index (implémentation directe, sans sklearn)."""
    a, b = np.asarray(a), np.asarray(b)
    ct = pd.crosstab(a, b).to_numpy()
    comb = lambda x: x * (x - 1) / 2
    sij, sa, sb = comb(ct).sum(), comb(ct.sum(1)).sum(), comb(ct.sum(0)).sum()
    n = comb(len(a))
    exp = sa * sb / n
    return (sij - exp) / (0.5 * (sa + sb) - exp)


tstars = [2, 5, 10, 20, 40]
ref, _ = clustering_at(TSTAR)
stab = pd.DataFrame({"t_star": tstars,
                     "ARI_vs_reference": [round(ari(ref, clustering_at(ts)[0]), 3) for ts in tstars]})
stab.to_csv(TABDIR / "clustering_stability.csv", index=False)
display(stab)

---
## 9. Validation statistique

Trois modèles nuls, tous exigés par un reviewer de *Scientific Reports* :

1. **Permutation des intensités** entre gènes (topologie conservée, biologie détruite) → la spécificité
   tissulaire du transport est-elle réelle ?
2. **Rewiring conservant les degrés** (biologie conservée, topologie détruite) → le résultat vient-il du
   graphe ou de l'expression ?
3. **Bootstrap sur les gènes** → intervalles de confiance sur la QTE.

Plus une **analyse de sensibilité** aux choix de modélisation (imputation, diagonale, normalisation).

In [ ]:
# ============================================================================
# 9.1 Test de permutation : intensités mélangées entre gènes
# ============================================================================
def permutation_test(tissue_idx, n_perm=None, rng=RNG):
    n_perm = CONFIG["N_PERM_CONTROL"] if n_perm is None else n_perm
    tt = TIMES[::5]            # grille grossiere : l'integrale est deja convergee
    obs = qte(WALKS[TISSUES[tissue_idx]], SOURCE, TARGET, tt)
    null = np.empty(n_perm)
    x = X[:, tissue_idx]
    for k in range(n_perm):
        Hp, _ = build_hamiltonian(M, rng.permutation(x))
        null[k] = qte(QuantumWalk(Hp), SOURCE, TARGET, tt)
    p = (1 + np.sum(np.abs(null - null.mean()) >= abs(obs - null.mean()))) / (n_perm + 1)
    z = (obs - null.mean()) / (null.std() + 1e-12)
    return obs, null, z, p


t0 = time.time()
perm_rows = []
for j, t in enumerate(TISSUES):
    obs, null, z, p = permutation_test(j)
    perm_rows.append({"tissue": t, "QTE_observed": obs, "null_mean": null.mean(),
                      "null_std": null.std(), "z_score": z, "p_value": p})
perm = pd.DataFrame(perm_rows)
# Correction de Benjamini-Hochberg (FDR)
o = np.argsort(perm["p_value"].to_numpy()); m = len(perm)
q = np.empty(m); q[o] = np.minimum.accumulate(
    (perm["p_value"].to_numpy()[o] * m / (np.arange(m) + 1))[::-1])[::-1]
perm["q_value_BH"] = np.clip(q, 0, 1)
perm.to_csv(TABDIR / "permutation_tests.csv", index=False)
print(f"{CONFIG['N_PERM_CONTROL']} permutations × {len(TISSUES)} tissus en {time.time()-t0:.1f} s")
display(perm.round(4))

In [ ]:
# ============================================================================
# 9.2 Modèle nul topologique : rewiring conservant les degrés
# ============================================================================
def rewire_membership(M, n_swaps=None, rng=RNG):
    """Échange d'appartenances gène-voie conservant le degré des gènes ET des voies
    (algorithme de double-edge swap sur le graphe biparti)."""
    Mr = M.copy()
    rows, cols = np.nonzero(Mr)
    n_swaps = n_swaps or 10 * len(rows)
    for _ in range(n_swaps):
        a, b = rng.integers(0, len(rows), 2)
        g1, p1, g2, p2 = rows[a], cols[a], rows[b], cols[b]
        if g1 == g2 or p1 == p2 or Mr[g1, p2] or Mr[g2, p1]:
            continue
        Mr[g1, p1] = Mr[g2, p2] = 0; Mr[g1, p2] = Mr[g2, p1] = 1
        cols[a], cols[b] = p2, p1
    return Mr


N_TOPO = 200   # plafonne : chaque rewiring exige ~17 000 echanges
j_ref = TISSUES.index(HIGH[0])
null_topo = []
for _ in range(N_TOPO):
    Mr = rewire_membership(M, n_swaps=5 * int(M.sum()))
    Hr, _ = build_hamiltonian(Mr, X[:, j_ref])
    null_topo.append(qte(QuantumWalk(Hr), SOURCE, TARGET))
null_topo = np.array(null_topo)
obs_topo = qte(WALKS[HIGH[0]], SOURCE, TARGET)
p_topo = (1 + (np.abs(null_topo - null_topo.mean()) >= abs(obs_topo - null_topo.mean())).sum()) / (N_TOPO + 1)
print(f"[{HIGH[0]}] QTE observée {obs_topo:.4f} vs nul topologique {null_topo.mean():.4f} "
      f"± {null_topo.std():.4f} — p = {p_topo:.4f}")

In [ ]:
# ============================================================================
# 9.3 Bootstrap sur les gènes -> IC 95 % de la QTE
# ============================================================================
def bootstrap_qte(tissue_idx, n_boot=CONFIG["N_BOOTSTRAP"], rng=RNG):
    vals = np.empty(n_boot)
    n_g = M.shape[0]
    for k in range(n_boot):
        idx = rng.integers(0, n_g, n_g)          # rééchantillonnage avec remise
        Hb, _ = build_hamiltonian(M[idx], X[idx, tissue_idx])
        vals[k] = qte(QuantumWalk(Hb), SOURCE, TARGET)
    return vals


boot_rows = []
for t in (HIGH[:2] + LOW[:2]):
    v = bootstrap_qte(TISSUES.index(t))
    boot_rows.append({"tissue": t, "QTE": wt[t], "ci_low": np.percentile(v, 2.5),
                      "ci_high": np.percentile(v, 97.5), "boot_sd": v.std()})
boot = pd.DataFrame(boot_rows)
boot.to_csv(TABDIR / "bootstrap_ci.csv", index=False)
display(boot.round(4))

In [ ]:
# ============================================================================
# 9.4 Analyse de sensibilité aux choix de modélisation
# ============================================================================
variants = {
    "reference":                dict(),
    "no on-site energies":           dict(include_diagonal=False),
    "no spectral normalisation":       dict(normalize="none"),
    "edges with >= 3 shared genes":        dict(min_shared=3),
    "background 0.5":           dict(_background=0.5),
}
sens_rows = []
for name, kw in variants.items():
    kw = dict(kw)
    bg = kw.pop("_background", None)
    Xv = np.where(X <= CONFIG["BACKGROUND"], bg, X) if bg else X
    for t in TISSUES:
        Hv, _ = build_hamiltonian(M, Xv[:, TISSUES.index(t)], **{**dict(
            include_diagonal=CONFIG["INCLUDE_DIAGONAL"], min_shared=CONFIG["MIN_SHARED_GENES"],
            normalize=CONFIG["NORMALIZE"]), **kw})
        sens_rows.append({"variant": name, "tissue": t, "QTE": qte(QuantumWalk(Hv), SOURCE, TARGET)})

# Variante topologique : uniquement les voies terminales (réseau non redondant)
pw_l, M_l, PW_l, LV_l = build_network(PW_ALL, keep_leaves_only=True)
src_l = find_pathway(CONFIG["SOURCE_PATHWAY"], PW_l, LV_l)
tgt_l = find_pathway(CONFIG["TARGET_PATHWAY"], PW_l, LV_l)
if src_l >= 0 and tgt_l >= 0:
    for t in TISSUES:
        Hl, _ = build_hamiltonian(M_l, X[:, TISSUES.index(t)])
        sens_rows.append({"variant": f"leaf pathways (n={len(PW_l)})", "tissue": t,
                          "QTE": qte(QuantumWalk(Hl), src_l, tgt_l)})
    print(f"Variante 'voies terminales' : {len(PW_l)} nœuds au lieu de {N}")
sens = pd.DataFrame(sens_rows)
wide = sens.pivot(index="tissue", columns="variant", values="QTE")
corr = wide.corr(method="spearman").round(3)
sens.to_csv(TABDIR / "sensitivity_analysis.csv", index=False)
print("A) Critère PAIRE (Complexe I → Complexe V) — Spearman du classement des tissus :")
display(corr)

# --- même analyse sur le critère GLOBAL (conductivité moyenne) --------------
gsens = []
for name, kw in variants.items():
    kw = dict(kw); bg = kw.pop("_background", None)
    Xv = np.where(X <= CONFIG["BACKGROUND"], bg, X) if bg else X
    for t in TISSUES:
        Hv, _ = build_hamiltonian(M, Xv[:, TISSUES.index(t)], **{**dict(
            include_diagonal=CONFIG["INCLUDE_DIAGONAL"], min_shared=CONFIG["MIN_SHARED_GENES"],
            normalize=CONFIG["NORMALIZE"]), **kw})
        gsens.append({"variant": name, "tissue": t,
                      "conductivity": avg_transfer_matrix(QuantumWalk(Hv), COARSE[::3])[off].mean()})
if src_l >= 0 and tgt_l >= 0:      # même variante topologique que ci-dessus
    for t in TISSUES:
        Hl, _ = build_hamiltonian(M_l, X[:, TISSUES.index(t)])
        gsens.append({"variant": f"leaf pathways (n={len(PW_l)})", "tissue": t,
                      "conductivity": mean_transfer_probability(QuantumWalk(Hl), COARSE[::3])})
gwide = pd.DataFrame(gsens).pivot(index="tissue", columns="variant", values="conductivity")
gwide.to_csv(TABDIR / "sensitivity_global.csv")
print("\nB) Critère GLOBAL (conductivité moyenne sur toutes les paires) :")
display(gwide.corr(method="spearman").round(3))
print("→ Comparez A et B : le critère dont les rho dépassent 0.8 doit servir de résultat principal ;\n"
      "   l'autre passe en analyse illustrative.")

In [ ]:
# --- Figure 8 : synthèse statistique ----------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.2))
axes[0].hist(null, bins=25, color="0.75", edgecolor="w")
axes[0].axvline(perm.loc[perm.tissue == TISSUES[-1], "QTE_observed"].iloc[0], color="crimson", lw=1.6)
axes[0].set_xlabel("Transfer efficiency under permutation"); axes[0].set_ylabel("Frequency")
axes[0].set_title("Biological null (last tissue)")

axes[1].hist(null_topo, bins=20, color="0.75", edgecolor="w")
axes[1].axvline(obs_topo, color="crimson", lw=1.6)
axes[1].set_xlabel("Transfer efficiency under rewiring"); axes[1].set_title(f"Topological null ({HIGH[0]})")

axes[2].errorbar(boot["QTE"], range(len(boot)),
                 xerr=[boot["QTE"] - boot["ci_low"], boot["ci_high"] - boot["QTE"]],
                 fmt="o", ms=4, color="0.2", capsize=3)
axes[2].set_yticks(range(len(boot))); axes[2].set_yticklabels(boot["tissue"])
axes[2].set_xlabel("Transfer efficiency (95% bootstrap CI)"); axes[2].set_title("Sampling uncertainty")
fig.suptitle("Statistical validation", y=1.05)
savefig(fig, "fig8_statistics", "Biological and topological null models, with bootstrap confidence intervals.")
plt.show()

---
### 9.5 Contrôles du critère principal

Quatre configurations, dont deux sans aucune valeur imputée. Un résultat qui survit aux quatre est
défendable en *peer review* ; un résultat qui n'apparaît que dans la configuration de référence doit
être abandonné. On vérifie aussi que le z-score reste décorrélé de la profondeur de détection dans
chaque configuration.


In [ ]:
# ============================================================================
# 9.5 Analyses de contrôle du critère principal — 4 configurations
# ============================================================================
# Objectif : montrer que le z de localisation ne dépend ni de l'imputation
# (complete-case), ni du terme diagonal (source du désordre d'Anderson).
NP_CTRL = CONFIG["N_PERM_CONTROL"]

# réseau restreint aux protéines détectées partout
M_CC = M[COMPLETE]
keep_cc = M_CC.sum(0) >= CONFIG["MIN_GENES_PER_PATHWAY"]
M_CC = M_CC[:, keep_cc]
print(f"Réseau complete-case : {M_CC.shape[0]} protéines, {M_CC.shape[1]} voies "
      f"(vs {M.shape[0]} / {M.shape[1]})")

CONFIGS = [
    ("reference (imputed, on-site)", M, X, True),
    ("no on-site energies", M, X, False),
    ("complete case", M_CC, X_CC_FULL, True),
    ("complete case, no on-site", M_CC, X_CC_FULL, False),
]

t0 = time.time()
ctrl = pd.concat([localization_zscores(Mx, Xx, include_diagonal=d, n_perm=NP_CTRL, label=name)
                  for name, Mx, Xx, d in CONFIGS], ignore_index=True)
ctrl.to_csv(TABDIR / "control_configurations.csv", index=False)
print(f"{len(CONFIGS)} configurations x {len(TISSUES)} tissus x {NP_CTRL} permutations "
      f"en {time.time()-t0:.0f} s")

zwide = ctrl.pivot(index="tissue", columns="config", values="z_localization")
qwide = ctrl.pivot(index="tissue", columns="config", values="q_value_BH")
display(zwide.round(2))

print("\nConcordance des classements (Spearman) :")
display(zwide.corr(method="spearman").round(3))

print("\nTissus significatifs (BH q<0.05) par configuration :")
for cfg in qwide.columns:
    s = qwide.index[qwide[cfg] < 0.05].tolist()
    print(f"  {cfg:32s} : {', '.join(s) if s else 'aucun'}")

robust = set(qwide.index[qwide[qwide.columns[0]] < 0.05])
for cfg in qwide.columns[1:]:
    robust &= set(qwide.index[qwide[cfg] < 0.05])
print(f"\n>>> Significatifs dans les QUATRE configurations : "
      f"{', '.join(sorted(robust)) if robust else 'aucun'}  <- affirmation publiable")

rho_min = zwide.corr(method="spearman").to_numpy()[np.triu_indices(len(zwide.columns), 1)].min()
print(f"Concordance minimale entre configurations : rho = {rho_min:.2f}")
if rho_min < 0.7:
    print("[!] Le CLASSEMENT fin des tissus n'est pas stable d'une configuration a l'autre.\n"
          "    Affirmation soutenable : 'la localisation depasse le nul apparie dans toutes les\n"
          "    configurations, et tel sous-ensemble de tissus est significatif partout'.\n"
          "    Affirmation NON soutenable : un ordre precis des 14 tissus.")

print("\nConfusion résiduelle avec la profondeur de détection :")
for cfg in zwide.columns:
    print(f"  {cfg:32s} : rho = {spearman(MISSING_FRAC, zwide.loc[TISSUES, cfg].to_numpy()):+.3f}")


In [ ]:
# --- Figure 10 : robustesse du critère principal -----------------------------
order = LOC["tissue"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))

w = 0.2
for i, cfg in enumerate(zwide.columns):
    axes[0].barh(np.arange(len(order)) + (i - 1.5) * w, zwide.loc[order, cfg],
                 height=w, label=cfg)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set_xlabel("Localisation $z$ (tissue-matched null)")
axes[0].legend(fontsize=6, loc="lower right"); axes[0].set_title("Robustness to modelling choices")

axes[1].scatter(100 * MISSING_FRAC, [LOC.set_index("tissue").loc[t, "return_prob"] for t in TISSUES],
                s=22, color="0.6", label="raw $R$")
ax2 = axes[1].twinx()
ax2.scatter(100 * MISSING_FRAC, [LOC.set_index("tissue").loc[t, "z_localization"] for t in TISSUES],
            s=22, color="crimson", marker="^", label="matched-null $z$")
axes[1].set_xlabel("Undetected proteins (%)"); axes[1].set_ylabel("Raw $R$ (confounded)")
ax2.set_ylabel("Matched-null $z$ (unconfounded)", color="crimson")
axes[1].set_title("Effect of detection depth")
fig.suptitle("Controls of the primary endpoint", y=1.04)
savefig(fig, "fig10_primary_controls",
        "Robustness of the localisation z-score across configurations, and absence of confounding with MS coverage.")
plt.show()


---
### 9.6–9.7 Représentation de l'abondance et schéma d'imputation — relecteur 3, points 3 et 4

Deux choix de modélisation susceptibles d'influer sur le spectre, donc sur la dynamique,
sont testés dans le cadre de permutation appariée, à nombre de permutations identique.


In [ ]:
# ============================================================================
# 9.6 Robustesse à la représentation de l'abondance — R3.3
# ============================================================================
# Le relecteur 3 note que l'emploi direct de l'intensité log10 comme poids modifie
# les éléments de matrice relatifs, donc le spectre, donc la dynamique. On teste
# quatre représentations défendables de la même mesure.
#
#   log10      : échelle du fichier source. Les intensités MS couvrent >7 ordres de
#                grandeur ; l'échelle log est la représentation standard en
#                protéomique quantitative et évite qu'une poignée de protéines
#                très abondantes ne domine entièrement tous les couplages.
#   linéaire   : 10^(log10). Donne tout le poids aux protéines les plus abondantes.
#   rang       : transformation en rangs normalisés (quantiles), indépendante de
#                toute hypothèse d'échelle.
#   z par tissu: centrée-réduite au sein de chaque tissu puis translatée pour
#                rester positive.
from scipy.stats import rankdata, mannwhitneyu, fisher_exact


def abundance_representations(Xs):
    reps = {"log10 (reference)": Xs.copy()}
    reps["linear intensity"] = np.power(10.0, Xs)
    rk = np.apply_along_axis(lambda c: rankdata(c) / (len(c) + 1), 0, Xs)
    reps["rank (quantile)"] = rk
    z = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-12)
    reps["z-score per tissue"] = z - z.min() + 0.1
    return reps


REPS = abundance_representations(X)
scale_rows = []
for name, Xr in REPS.items():
    zz = localization_zscores(M, Xr, n_perm=CONFIG["N_PERM_CONTROL"], label=name)
    scale_rows.append(zz)
scale = pd.concat(scale_rows, ignore_index=True)
scale.to_csv(TABDIR / "abundance_scale_robustness.csv", index=False)

szw = scale.pivot(index="tissue", columns="config", values="z_localization")
sqw = scale.pivot(index="tissue", columns="config", values="q_value_BH")
display(szw.round(2))

print("Concordance des classements tissulaires entre représentations (Spearman) :")
display(szw.corr(method="spearman").round(3))
print("\nTissus significatifs (BH q < 0.05) par représentation :")
for c in sqw.columns:
    s = sqw.index[sqw[c] < 0.05].tolist()
    print(f"  {c:22s} {len(s):2d}/{len(TISSUES)}  {', '.join(s) if s else 'aucun'}")
print(f"\nSigne de z positif partout : {bool((szw > 0).all().all())}")


In [ ]:
# ============================================================================
# 9.7 Sensibilité au schéma d'imputation — R3.4
# ============================================================================
# Les cinq schémas du §1.6 sont passés dans le cadre de permutation appariée,
# à nombre de permutations identique.
imp_rows = []
for name, Xi in IMPUTATIONS.items():
    imp_rows.append(localization_zscores(M, Xi, n_perm=CONFIG["N_PERM_CONTROL"], label=name))
impz = pd.concat(imp_rows, ignore_index=True)
impz.to_csv(TABDIR / "imputation_robustness.csv", index=False)

izw = impz.pivot(index="tissue", columns="config", values="z_localization")[list(IMPUTATIONS)]
iqw = impz.pivot(index="tissue", columns="config", values="q_value_BH")[list(IMPUTATIONS)]
display(izw.round(2))

print("Tissus significatifs (BH q < 0.05) par schéma :")
for c in izw.columns:
    print(f"  {c:18s} {int((iqw[c] < 0.05).sum()):2d}/{len(TISSUES)}")
print("\nConcordance des classements (Spearman) :")
display(izw.corr(method="spearman").round(3))

principled = [c for c in izw.columns if c != "legacy 0.1"]
sub = izw[principled].corr(method="spearman").to_numpy()
print(f"\nEntre les quatre schémas fondés sur la limite de détection observée, la concordance")
print(f"minimale vaut rho = {sub[np.triu_indices(len(principled),1)].min():.3f}.")
print(f"Le schéma « legacy 0.1 » s'en écarte "
      f"(rho max = {izw.corr(method='spearman').loc['legacy 0.1', principled].max():.3f}) ET donne")
print("les z les plus FAIBLES : l'ancienne imputation sous-estimait l'effet au lieu de le créer.")
print("\nConfusion résiduelle avec la profondeur de détection, par schéma :")
for c in izw.columns:
    r = spearman(MISSING_FRAC, izw.loc[TISSUES, c].to_numpy())
    n = len(TISSUES); tstat = r * np.sqrt((n - 2) / max(1 - r**2, 1e-12))
    from scipy import stats as _st
    pv = 2 * (1 - _st.t.cdf(abs(tstat), n - 2))
    print(f"  {c:18s} rho = {r:+.3f}  (p = {pv:.3f})")


In [ ]:
# --- Figure 13 : robustesse du critère principal aux choix de modélisation
def _pretty(t):
    return (t.replace("skeletalmuscle", "skeletal muscle").replace("smallintestine", "small intestine")
             .replace("largeintestine", "large intestine").replace("spinalcord", "spinal cord"))


fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.9), gridspec_kw={"wspace": 0.30})
order = LOC["tissue"].tolist()
labels = [_pretty(t) for t in order]

# (a) schéma d'imputation ----------------------------------------------------
for c in izw.columns:
    axes[0].plot(izw.loc[order, c], range(len(order)), marker="o", ms=3.4, lw=1, label=c)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set_xlabel("Matched-null $z$")
axes[0].set_title("a   Imputation scheme", loc="left", fontsize=9)
axes[0].legend(fontsize=6, loc="lower right", framealpha=0.9)

# (b) représentation de l'abondance -----------------------------------------
for c in szw.columns:
    axes[1].plot(szw.loc[order, c], range(len(order)), marker="s", ms=3.4, lw=1, label=c)
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set_xlabel("Matched-null $z$")
axes[1].set_title("b   Abundance representation", loc="left", fontsize=9)
axes[1].legend(fontsize=6, loc="lower right", framealpha=0.9)

# (c) fenêtre d'observation --------------------------------------------------
# R dépend fortement de T en valeur absolue ; ce qui importe est la STABILITÉ DU
# CLASSEMENT des tissus. On centre-réduit donc R au sein de chaque fenêtre.
wz = wwide.loc[order].apply(lambda col: (col - col.mean()) / (col.std() + 1e-12))
wcols = sorted(wwide.columns, key=lambda c: (float(c.split("=")[1].split(",")[0]),
                                             int(c.split("n=")[1])))
for c in wcols:
    axes[2].plot(wz[c], range(len(order)), marker="^", ms=3.4, lw=1, label=c)
axes[2].axvline(0, color="k", lw=0.8)
axes[2].set_xlabel("Standardised $R$ within window")
axes[2].set_title("c   Observation window", loc="left", fontsize=9)
axes[2].legend(fontsize=6, loc="lower right", framealpha=0.9)

for ax in axes:
    ax.set_yticks(range(len(order)))
    ax.invert_yaxis()
    ax.margins(y=0.03)
axes[0].set_yticklabels(labels, fontsize=7)
for ax in axes[1:]:
    ax.set_yticklabels([])

fig.suptitle("Robustness of the primary endpoint to modelling choices", y=1.02, fontsize=10)
savefig(fig, "fig13_robustness_panel",
        "Stability of the localisation endpoint across imputation schemes, abundance "
        "representations and observation windows.")
plt.show()


---
## 9.8–9.10 Comparaison à des méthodes non quantiques — relecteur 2

Le relecteur 2 pose la question centrale : qu'apporte la représentation par marche quantique
au-delà d'analyses classiques ou plus simples ? Il demande une comparaison sur les mêmes
données et selon les mêmes critères, pour deux tâches précises — distinguer les tissus et
identifier les gènes de maladie.

C'est l'ajout le plus important de cette révision. Neuf scores concurrents sont évalués,
dont un seul est quantique, et le résultat est rapporté tel quel, qu'il soit favorable
ou non à la méthode proposée.


In [ ]:
# ============================================================================
# 9.8 Comparaison à des méthodes NON QUANTIQUES — tâche A : distinguer les tissus
# ============================================================================
# Demande centrale du relecteur 2 : démontrer ce que la représentation par marche
# quantique apporte au-delà d'analyses plus simples ou classiques, sur les MÊMES
# données et selon les MÊMES critères.
#
# Quatre statistiques sont donc évaluées dans un cadre rigoureusement identique :
# même réseau, mêmes intensités, même nul apparié au tissu, même nombre de
# permutations, même correction de multiplicité.
#
#   CTQW         : probabilité de retour moyenne de la marche quantique   (proposée)
#   CTRW         : probabilité de retour moyenne de la marche classique   (analogue direct)
#   Degré pondéré: variance du degré pondéré du graphe, sans dynamique     (réseau simple)
#   Gini         : coefficient de Gini de l'abondance, sans réseau du tout (données brutes)
def classical_return(L_or_H, times):
    """Probabilité de retour moyenne de la marche classique p(t) = exp(-Lt)."""
    A = np.abs(L_or_H).copy(); np.fill_diagonal(A, 0.0)
    Lp = np.diag(A.sum(1)) - A
    s = np.abs(np.linalg.eigvalsh(Lp)).max()
    Lp = Lp / s if s > 0 else Lp
    ev, V = np.linalg.eigh(Lp)
    V2 = V ** 2
    acc = 0.0
    for t in times:
        acc += V2 @ np.exp(-ev * t)
    return float(np.mean(acc / len(times)))


def gini(v):
    v = np.sort(np.asarray(v, float)); n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1) @ v / (n * v.sum() + 1e-12))


def stat_ctqw(Mx, x, times):
    H, _ = build_hamiltonian(Mx, x); return mean_return_probability(QuantumWalk(H), times)


def stat_ctrw(Mx, x, times):
    H, _ = build_hamiltonian(Mx, x); return classical_return(H, times)


def stat_degvar(Mx, x, times=None):
    H, _ = build_hamiltonian(Mx, x)
    A = np.abs(H).copy(); np.fill_diagonal(A, 0.0)
    d = A.sum(1); return float(d.var() / (d.mean() ** 2 + 1e-12))


def stat_gini(Mx, x, times=None):
    return gini(np.asarray(x, float))


METHODS_A = {"CTQW return probability": stat_ctqw,
             "CTRW return probability": stat_ctrw,
             "Weighted-degree variance": stat_degvar,
             "Abundance Gini (no network)": stat_gini}


def matched_null_z(statfn, Mx, Xx, n_perm, times=None, seed=None):
    """Exactement le même cadre de permutation appariée pour toute statistique."""
    times = FAST_T if times is None else times
    rng = np.random.default_rng(CONFIG["SEED"] if seed is None else seed)
    rows = []
    for j, t in enumerate(TISSUES):
        obs = statfn(Mx, Xx[:, j], times)
        null = np.array([statfn(Mx, rng.permutation(Xx[:, j]), times) for _ in range(n_perm)])
        p = (1 + np.sum(np.abs(null - null.mean()) >= abs(obs - null.mean()))) / (n_perm + 1)
        rows.append({"tissue": t, "statistic": obs, "z": (obs - null.mean()) / (null.std() + 1e-12),
                     "p": p})
    out = pd.DataFrame(rows); out["q_BH"] = bh(out["p"])
    return out


t0 = time.time()
benchA = []
for name, fn in METHODS_A.items():
    r = matched_null_z(fn, M, X, CONFIG["N_PERM_BENCHMARK"])
    r.insert(0, "method", name)
    benchA.append(r)
benchA = pd.concat(benchA, ignore_index=True)
benchA.to_csv(TABDIR / "benchmark_taskA_tissue.csv", index=False)
print(f"Tâche A terminée en {time.time()-t0:.0f} s "
      f"({CONFIG['N_PERM_BENCHMARK']} permutations x {len(TISSUES)} tissus x {len(METHODS_A)} méthodes)")

bAz = benchA.pivot(index="tissue", columns="method", values="z")
bAq = benchA.pivot(index="tissue", columns="method", values="q_BH")
display(bAz.round(2))

summaryA = pd.DataFrame({
    "method": list(METHODS_A),
    "n_significant_q05": [int((bAq[m] < 0.05).sum()) for m in METHODS_A],
    "median_abs_z": [float(np.abs(bAz[m]).median()) for m in METHODS_A],
    "rho_vs_detection_depth": [spearman(MISSING_FRAC, bAz.loc[TISSUES, m].to_numpy()) for m in METHODS_A],
    "rho_vs_CTQW_ranking": [spearman(bAz["CTQW return probability"], bAz[m]) for m in METHODS_A],
})
summaryA.to_csv(TABDIR / "benchmark_taskA_summary.csv", index=False)
display(summaryA.round(3))
print("Lecture. Si la CTRW classique atteint la même sensibilité et le même classement que la")
print("CTQW, la représentation quantique n'apporte rien sur cette tâche, et il faut le dire.")
print("\nLe Gini de l'abondance est un CONTRÔLE NÉGATIF par construction : il ne dépend que de la")
print("distribution marginale des intensités, que la permutation appariée conserve exactement.")
print("Son z est donc nul à la précision machine. Cela vérifie que le nul apparié teste bien")
print("l'AFFECTATION de l'abondance aux positions du réseau, et non la distribution elle-même.")


In [ ]:
# ============================================================================
# 9.9 Comparaison à des méthodes NON QUANTIQUES — tâche B : identifier les gènes
#     de maladie mitochondriale — R2.2
# ============================================================================
# Même jeu de protéines (878), même liste de référence (195 gènes), mêmes critères
# d'évaluation (AUROC et AUPRC), pour neuf scores dont un seul est quantique.
import networkx as nx
from sklearn.metrics import roc_auc_score, average_precision_score

G_nx = nx.from_numpy_array(ADJ)
BETWEEN = np.array([v for _, v in sorted(nx.betweenness_centrality(G_nx).items())])
PAGERANK = np.array([v for _, v in sorted(nx.pagerank(G_nx).items())])
EIGCEN = np.array([v for _, v in sorted(nx.eigenvector_centrality_numpy(G_nx).items())])


def classical_return_from_H(H, times):
    A = np.abs(H).copy(); np.fill_diagonal(A, 0.0)
    Lp = np.diag(A.sum(1)) - A
    s = np.abs(np.linalg.eigvalsh(Lp)).max()
    Lp = Lp / s if s > 0 else Lp
    ev, V = np.linalg.eigh(Lp); V2 = V ** 2
    acc = 0.0
    for t in times:
        acc += V2 @ np.exp(-ev * t)
    return float(np.mean(acc / len(times)))


t0 = time.time()
bench_rows = []
for t in SCAN_TISSUES:
    j = TISSUES.index(t)
    H0, _ = build_hamiltonian(M, X[:, j])
    Rq0 = mean_return_probability(QuantumWalk(H0), FAST_T)
    Rc0 = classical_return_from_H(H0, FAST_T)
    for gi in in_network:
        x = X[:, j].copy(); x[gi] = X[:, j].min()
        Hk, _ = build_hamiltonian(M, x)
        bench_rows.append({
            "gene": GENES[gi], "tissue": t,
            "CTQW knockout |dR|": abs(mean_return_probability(QuantumWalk(Hk), FAST_T) - Rq0) / Rq0,
            "CTRW knockout |dR|": abs(classical_return_from_H(Hk, FAST_T) - Rc0) / Rc0,
            "Pathway membership count": M[gi].sum(),
            "Summed pathway degree": float(M[gi] @ PATH_DEGREE),
            "Summed betweenness": float(M[gi] @ BETWEEN),
            "Summed PageRank": float(M[gi] @ PAGERANK),
            "Summed eigenvector centrality": float(M[gi] @ EIGCEN),
            "Mean protein abundance": float(X[gi].mean()),
            "Tissue protein abundance": float(X[gi, j]),
        })
benchB = pd.DataFrame(bench_rows)
benchB["is_disease_gene"] = benchB["gene"].isin(DISEASE_GENES)
benchB.to_csv(TABDIR / "benchmark_taskB_scores.csv", index=False)
print(f"Tâche B : {len(in_network)} protéines x {len(SCAN_TISSUES)} tissus x 9 scores "
      f"en {time.time()-t0:.0f} s")

SCORES_B = [c for c in benchB.columns if c not in ("gene", "tissue", "is_disease_gene")]
rows = []
for t in SCAN_TISSUES:
    s = benchB[benchB.tissue == t]
    y = s["is_disease_gene"].astype(int).to_numpy()
    for sc in SCORES_B:
        v = s[sc].to_numpy(float)
        rows.append({"tissue": t, "score": sc,
                     "AUROC": roc_auc_score(y, v), "AUPRC": average_precision_score(y, v)})
perf = pd.DataFrame(rows)
perf.to_csv(TABDIR / "benchmark_taskB_performance.csv", index=False)

auroc = perf.pivot(index="score", columns="tissue", values="AUROC")
auprc = perf.pivot(index="score", columns="tissue", values="AUPRC")
auroc["mean"] = auroc.mean(1); auprc["mean"] = auprc.mean(1)
auroc = auroc.sort_values("mean", ascending=False)
print("\nAUROC par score et par tissu (prévalence des gènes de maladie : "
      f"{100*benchB.groupby('tissue').is_disease_gene.mean().iloc[0]:.1f} %) :")
display(auroc.round(3))
print("\nAUPRC :")
display(auprc.loc[auroc.index].round(3))


In [ ]:
# ============================================================================
# 9.10 La CTQW bat-elle son analogue classique ? Test apparié sur l'AUROC — R2.2
# ============================================================================
# L'écart d'AUROC entre la CTQW et chaque comparateur est évalué par bootstrap
# stratifié apparié (mêmes protéines rééchantillonnées pour les deux scores), ce qui
# tient compte de la corrélation entre scores.
def auroc_delta_bootstrap(s, score_a, score_b, n_boot=CONFIG["N_BOOTSTRAP_AUC"], seed=0):
    y = s["is_disease_gene"].astype(int).to_numpy()
    a, b = s[score_a].to_numpy(float), s[score_b].to_numpy(float)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    rng = np.random.default_rng(seed)
    d = np.empty(n_boot)
    for k in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)])
        yy = y[idx]
        d[k] = roc_auc_score(yy, a[idx]) - roc_auc_score(yy, b[idx])
    obs = roc_auc_score(y, a) - roc_auc_score(y, b)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return obs, np.percentile(d, 2.5), np.percentile(d, 97.5), max(p, 1.0 / n_boot)


REF = "CTQW knockout |dR|"
cmp_rows = []
for t in SCAN_TISSUES:
    s = benchB[benchB.tissue == t]
    for sc in SCORES_B:
        if sc == REF:
            continue
        obs, lo, hi, p = auroc_delta_bootstrap(s, REF, sc, seed=CONFIG["SEED"] % 10000)
        cmp_rows.append({"tissue": t, "comparator": sc, "delta_AUROC": obs,
                         "CI95_low": lo, "CI95_high": hi, "p_bootstrap": p})
cmpB = pd.DataFrame(cmp_rows)
cmpB["q_BH"] = bh(cmpB["p_bootstrap"])
cmpB.to_csv(TABDIR / "benchmark_taskB_auroc_deltas.csv", index=False)
display(cmpB.round(4))

wins = cmpB[(cmpB.delta_AUROC > 0) & (cmpB.q_BH < 0.05)]
loss = cmpB[(cmpB.delta_AUROC < 0) & (cmpB.q_BH < 0.05)]
print(f"\nLa CTQW surpasse significativement le comparateur dans {len(wins)} cas sur {len(cmpB)} ;")
print(f"elle lui est significativement inférieure dans {len(loss)} cas.")
if len(loss):
    print("Comparateurs qui battent la CTQW :",
          ", ".join(sorted(set(loss.comparator))))
print("\nCe tableau est la réponse directe au relecteur 2 : il indique, sur les mêmes données et")
print("selon le même critère, si la représentation quantique apporte un gain mesurable. Le")
print("résultat est rapporté tel quel, qu'il soit favorable ou non.")


In [ ]:
# --- Figure 12 : comparaison aux méthodes non quantiques (tâches A et B)
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.0),
                         gridspec_kw={"width_ratios": [1.0, 1.0], "wspace": 0.52})

# ---- Tâche A : séparer les tissus de leur nul ------------------------------
order = LOC["tissue"].tolist()[::-1]
w = 0.2
for i, m in enumerate(METHODS_A):
    axes[0].barh(np.arange(len(order)) + (i - 1.5) * w, bAz.loc[order, m], height=w, label=m)
axes[0].set_yticks(range(len(order)))
axes[0].set_yticklabels([t.replace("skeletalmuscle", "skeletal muscle")
                          .replace("smallintestine", "small intestine")
                          .replace("largeintestine", "large intestine")
                          .replace("spinalcord", "spinal cord") for t in order], fontsize=7)
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set_xlabel("Matched-null $z$")
axes[0].set_title("a   Task A — separating tissues from their null", loc="left", fontsize=9)
axes[0].legend(fontsize=6, loc="lower right", framealpha=0.9)
axes[0].margins(y=0.02)

# ---- Tâche B : classer les gènes de maladie --------------------------------
ord_b = auroc.index.tolist()
ypos = np.arange(len(ord_b))
for i, t in enumerate(SCAN_TISSUES):
    axes[1].scatter(auroc.loc[ord_b, t], ypos + (i - 1) * 0.18, s=26, label=t, zorder=3)
axes[1].axvline(0.5, color="k", ls=":", lw=1)
axes[1].set_yticks(ypos)
axes[1].set_yticklabels(ord_b, fontsize=7)
# étiquettes à droite : elles ne peuvent plus empiéter sur le panneau de gauche
axes[1].yaxis.tick_right()
axes[1].yaxis.set_label_position("right")
axes[1].tick_params(axis="y", length=0, pad=4)
axes[1].set_xlabel("AUROC for mitochondrial disease genes")
axes[1].set_title("b   Task B — ranking disease genes", loc="left", fontsize=9)
axes[1].legend(fontsize=7, loc="lower right", framealpha=0.9)
axes[1].invert_yaxis()
axes[1].margins(y=0.04)

fig.suptitle("Quantum walk against non-quantum comparators on the same data", y=1.02, fontsize=10)
savefig(fig, "fig12_benchmark",
        "Comparison of the continuous-time quantum walk with classical and purely structural "
        "methods on tissue separation and disease-gene ranking.")
plt.show()


---
## 10. Export, reproductibilité et matériel supplémentaire

In [ ]:
# ============================================================================
# 10.1 Journal de session + index des figures + archive des résultats
# ============================================================================
session = {
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {p.__name__: p.__version__ for p in (np, pd, scipy, mpl)},
    "config": CONFIG,
    "data": {**PROVENANCE, "n_genes": int(len(GENES)),
             "n_pathways": int(N), "tissues": TISSUES,
             "source_pathway": PATHWAYS[SOURCE], "target_pathway": PATHWAYS[TARGET]},
    "seed": CONFIG["SEED"],
}
(OUT / "session_info.json").write_text(json.dumps(session, indent=2, default=str))
pd.DataFrame(FIGURE_INDEX).to_csv(TABDIR / "figure_index.csv", index=False)

print(json.dumps(session["data"], indent=2))
print("\nFichiers générés :")
for f in sorted(OUT.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT)}  ({f.stat().st_size/1024:.1f} Ko)")

In [ ]:
# ============================================================================
# 10.2 Table 1 formatée pour le manuscrit (LaTeX + Markdown)
# ============================================================================
table1 = metrics[["tissue", "QTE_quantum", "QTE_classical", "quantum_advantage",
                  "mixing_time", "mean_IPR", "spectral_gap"]].round(4)
table1.columns = ["Tissue", "QTE (quantum)", "QTE (classical)", "Quantum advantage",
                  r"$\tau_{mix}$", "Mean IPR", "Spectral gap"]
(OUT / "table1.tex").write_text(table1.to_latex(index=False, escape=False,
                                                caption="Tissue-specific quantum transport metrics.",
                                                label="tab:metrics"))
try:
    (OUT / "table1.md").write_text(table1.to_markdown(index=False))   # nécessite `tabulate`
except ImportError:
    (OUT / "table1.csv").write_text(table1.to_csv(index=False))
display(table1)

---
## 11. Correspondance avec le manuscrit révisé

| Sortie | Section du manuscrit |
|---|---|
| `imputation_schemes.csv`, `imputation_robustness.csv` | Methods 2.1, Results 3.7 |
| `localization_primary.csv` | Table 2, Results 3.5 |
| `localization_decomposition.csv`, `anderson_disorder_scaling.csv`, `fig11` | Results 3.6 |
| `observation_window_stability.csv` | Methods 2.4, Supplementary |
| `lindblad_dt_convergence.csv`, `lindblad_T_convergence.csv` | Methods 2.4, Supplementary |
| `enrichment_adjusted_logistic.csv`, `enrichment_stratified_permutation.csv` | Table 5, Results 3.9 |
| `abundance_scale_robustness.csv`, `fig13` | Results 3.7 |
| `benchmark_taskA_*.csv`, `benchmark_taskB_*.csv`, `fig12` | Table 6, Results 3.10 |

### Avant resoumission

1. Déposer ce notebook et les tables sur Zenodo et remplacer le DOI dans *Code availability*
   (relecteur 3, point 12).
2. Remplacer `DISEASE_GENES` par un export daté de MITOMAP ou OMIM et citer la version.
3. Vérifier que la terminologie neutre est appliquée uniformément dans le manuscrit
   (relecteur 3, point 11).
